#  WIPO IP Data Downloader

This script downloads **Patent, Trademark, and Industrial Design** statistics directly from the WIPO database. 

###  Configuration
You can adjust the settings in **Section 1** of the code below:
* **`FROM_YEAR` / `TO_YEAR`**: The date range for the data (e.g., `1980` to `2024`).
* **`OUTPUT_DIR`**: The folder where `.csv` files will be saved (default: `wipo_data_downloads`).
* **`MAX_WORKERS`**: How many files to download at the same time (default: `8`).

###  Instructions
1.  **Paste your URLs**: Go to the `URL_TEMPLATES` dictionary in the code and paste the API links from the WIPO Data Catalogue.
2.  **Run the Cell**: The script will automatically:
    * Handle authentication cookies.
    * Retry failed downloads.
    * Rename files to be human-readable (e.g., `patent_1a...csv`).

In [1]:
import os
import requests
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

In [2]:
class UrlSpec:
    def __init__(self, url_template=""):
        self.url_template = url_template

In [ ]:
# Configuration
from_year = 1980
to_year = 2024

MAX_WORKERS = 1
OUTPUT_DIR = "or_country_profiles-main\data\raw\wipo\ip_indicators"

In [5]:
print(f" Calculated Output Path: {OUTPUT_DIR}")

aw\wipo\ip_indicatorsth: or_country_profiles-main\data


In [ ]:
# Data Mapping 
# Map: KEY -> Original WIPO Filename Template
FILENAME_MAP = {

    ##################### INDUSTRIAL DESIGNS ###########################################################################################################
    # fct_ip_flows_yearly
    "ID1a": "industrial_1a- Direct applications_Count by filing office and applicant's origin_{from_year}_{to_year}.csv",
    "ID1b": "industrial_1b- Applications via the Hague system_Count by filing office and applicant's origin_{from_year}_{to_year}.csv",
    "ID2a": "industrial_2a- Registrations for direct applications_Count by filing office and applicant's origin_{from_year}_{to_year}.csv",
    "ID2b": "industrial_2b- Registrations for applications via the Hague system_Count by filing office and applicant's origin_{from_year}_{to_year}.csv",
    "ID7":  "industrial_7 - Design registrations in force (by office)_Total count by filing office_{from_year}_{to_year}.csv",

    #fct_ip_flows_by_class_yearly
    "ID3a": "industrial_3a- Direct applications by class_Count by filing office and applicant's origin_{from_year}_{to_year}.csv",
    "ID3b": "industrial_3b- Applications via the Hague system by class_Count by filing office and applicant's origin_{from_year}_{to_year}.csv",
    "ID4a": "industrial_4a- Registrations for direct applications by class_Count by filing office and applicant's origin_{from_year}_{to_year}.csv",
    "ID4b": "industrial_4b- Registrations for applications via the Hague system by class_Count by filing office and applicant's origin_{from_year}_{to_year}.csv",

    #fct_ip_flows_by_residency
    "ID1a_origin": "industrial_1a_origin- Direct applications_Totals by origin and residency type_{from_year}_{to_year}.csv",
    "ID1b_origin": "industrial_1b_origin- Applications via the Hague system_Totals by origin and residency type_{from_year}_{to_year}.csv",
    "ID1a_office": "industrial_1a_office- Direct applications_Totals by office and residency type_{from_year}_{to_year}.csv",
    "ID1b_office": "industrial_1b_office- Applications via the Hague system_Totals by office and residency type_{from_year}_{to_year}.csv",

    "ID2a_origin": "industrial_2a_origin- Registrations for direct applications_Totals by origin and residency type_{from_year}_{to_year}.csv",
    "ID2b_origin": "industrial_2b_origin- Registrations for applications via the Hague system_Totals by origin and residency type_{from_year}_{to_year}.csv",
    "ID2a_office": "industrial_2a_office- Registrations for direct applications_Totals by office and residency type_{from_year}_{to_year}.csv",
    "ID2b_office": "industrial_2b_office- Registrations for applications via the Hague system_Totals by office and residency type_{from_year}_{to_year}.csv",




   

    ##################### PATENTS ########################################################################################################################
    # fct_ip_flows_yearly
    "PA1a": "patent_1a- Direct applications_Count by filing office and applicant's origin_{from_year}_{to_year}.csv",
    "PA1b": "patent_1b- PCT national phase entries_Count by filing office and applicant's origin_{from_year}_{to_year}.csv",
    "PA2a": "patent_2a- Grant for direct applications_Count by filing office and applicant's origin_{from_year}_{to_year}.csv",
    "PA2b": "patent_2b- Grant for PCT national phase entries_Count by filing office and applicant's origin_{from_year}_{to_year}.csv",
    "PA3":  "patent_3 - Patents in force_Total count by filing office_{from_year}_{to_year}.csv",

    #fct_ip_flows_by_class_yearly
    "PA4a": "patent_4a - Patent publications by technology_Count by filling office and applicant's origin_{from_year}_{to_year}.csv",
    "PA5":  "patent_5 - Patent grants by technology_Count by filing office and applicant's origin_{from_year}_{to_year}.csv",

    #fct_ip_flows_by_residency
    "PA1a_origin": "patent_1a_origin- Direct applications_Totals by origin and residency type_{from_year}_{to_year}.csv",
    "PA1b_origin": "patent_1b_origin- PCT national phase entries_Totals by origin and residency type_{from_year}_{to_year}.csv",
    "PA1a_office": "patent_1a_office- Direct applications_Totals by office and residency type_{from_year}_{to_year}.csv",
    "PA1b_office": "patent_1b_office- PCT national phase entries_Totals by office and residency type_{from_year}_{to_year}.csv",

    "PA2a_origin": "patent_2a_origin- Grants for direct applications_Totals by origin and residency type_{from_year}_{to_year}.csv",
    "PA2b_origin": "patent_2b_origin- Grants for PCT national phase entries_Totals by origin and residency type_{from_year}_{to_year}.csv",
    "PA2a_office": "patent_2a_office- Grants for direct applications_Totals by office and residency type_{from_year}_{to_year}.csv",
    "PA2b_office": "patent_2b_office- Grants for PCT national phase entries_Totals by office and residency type_{from_year}_{to_year}.csv",






    ##################### TRADEMARKS ######################################################################################################################
    # fct_ip_flows_yearly
    "TM1a": "trademark_1a- Direct applications_Count by filing office and applicant's origin_{from_year}_{to_year}.csv",
    "TM1b": "trademark_1b- Applications via the Madrid system_Count by filing office and applicant's origin_{from_year}_{to_year}.csv",
    "TM2a": "trademark_2a- Registrations for direct applications_Count by filing office and applicant's origin_{from_year}_{to_year}.csv",
    "TM2b": "trademark_2b- Registrations for applications via the Madrid system_Count by filing office and applicant's origin_{from_year}_{to_year}.csv",
    "TM7":  "trademark_7 - Trademarks in force (by office)_Total count by filing office_{from_year}_{to_year}.csv",


    #fct_ip_flows_by_class_yearly
    "TM3a": "trademark_3a- Direct applications by class_Count by filing office and applicant's origin_{from_year}_{to_year}.csv",
    "TM3b": "trademark_3b- Applications via the Madrid system by class_Count by filing office and applicant's origin_{from_year}_{to_year}.csv",
    "TM4a": "trademark_4a- Registrations for direct applications by class_Count by filing office and applicant's origin_{from_year}_{to_year}.csv",
    "TM4b": "trademark_4b- Registrations for applications via the Madrid system by class_Count by filing office and applicant's origin_{from_year}_{to_year}.csv",

    #fct_ip_flows_by_residency
    "TM1a_origin": "trademark_1a_origin- Direct applications_Totals by origin and residency type_{from_year}_{to_year}.csv",
    "TM1b_origin": "trademark_1b_origin- Applications via the Madrid system_Totals by origin and residency type_{from_year}_{to_year}.csv",
    "TM1a_office": "trademark_1a_office- Direct applications_Totals by office and residency type_{from_year}_{to_year}.csv",
    "TM1b_office": "trademark_1b_office- Applications via the Madrid system_Totals by office and residency type_{from_year}_{to_year}.csv",

    "TM2a_origin": "trademark_2a_origin- Registrations for direct applications_Totals by origin and residency type_{from_year}_{to_year}.csv",
    "TM2b_origin": "trademark_2b_origin- Registrations for applications via the Madrid system_Totals by origin and residency type_{from_year}_{to_year}.csv",
    "TM2a_office": "trademark_2a_office- Registrations for direct applications_Totals by office and residency type_{from_year}_{to_year}.csv",
    "TM2b_office": "trademark_2b_office- Registrations for applications via the Madrid system_Totals by office and residency type_{from_year}_{to_year}.csv",

}

In [14]:
# Helper Functions

def get_session():
    """Creates a robust session with browser-like headers."""
    s = requests.Session()
    retry = Retry(
        total=5, connect=3, read=3, backoff_factor=1,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"], raise_on_status=False
    )
    adapter = HTTPAdapter(max_retries=retry)
    s.mount("https://", adapter)
    
    # Updated Headers to look like a modern browser visiting the main page
    s.headers.update({
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        "Origin": "https://www3.wipo.int",
        "Referer": "https://www3.wipo.int/ipstats/index.htm",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.9",
        "Accept-Encoding": "gzip, deflate, br",
        "Connection": "keep-alive",
        "Upgrade-Insecure-Requests": "1"
    })
    return s

def prime_cookies(session):
    print("Priming cookies (visiting homepage)...")
    try:
        # Visit the main page first to get a valid session ID
        session.get("https://www3.wipo.int/ipstats/index.htm", timeout=30)
    except Exception as e:
        print(f"Warning: Cookie priming failed: {e}")

def safe_filename(name):
    return "".join(ch for ch in name if ch not in '<>:"/\\|?*').strip()

def build_url(template_input, f_year, t_year):
    # Handle UrlSpec object vs String
    if hasattr(template_input, "url_template"):
        template = template_input.url_template
    else:
        template = str(template_input)

    if not template: 
        return ""

    if "{from_year}" in template:
        return template.format(from_year=f_year, to_year=t_year)
    
    # Ensure we don't create double && or ?&
    joiner = "&" if "?" in template else "?"
    if template.endswith("?") or template.endswith("&"):
        joiner = ""
        
    return f"{template}{joiner}fromYear={f_year}&toYear={t_year}"

def download_task(session, key, url_input, out_dir):
    url = build_url(url_input, FROM_YEAR, TO_YEAR)
    if not url: return None 

    # Determine Filename
    if key in FILENAME_MAP:
        fname = FILENAME_MAP[key].format(from_year=FROM_YEAR, to_year=TO_YEAR)
    else:
        fname = f"{key}_{from_year}_{to_year}.csv"
    
    fname = safe_filename(fname)
    fpath = out_dir / fname

    # Check if exists
    if fpath.exists() and fpath.stat().st_size > 0:
        return f"Skipped (Exists): {fname}"

    # Request
    try:
        # Standard GET request
        with session.get(url, timeout=300, stream=True) as r:
            
            # If 400, try forcing a specific CSV download hint if not already there
            if r.status_code == 400:
                print(f" 400 Error for {key}. Retrying with downloadType=CSV...")
                if "downloadType" not in url:
                    retry_url = url + ("&" if "?" in url else "?") + "downloadType=CSV"
                    r.close()
                    r = session.get(retry_url, timeout=300, stream=True)
            
            if r.status_code != 200:
                raise RuntimeError(f"HTTP {r.status_code} - {r.reason}")

            tmp_path = fpath.with_suffix(".part")
            with open(tmp_path, "wb") as f:
                for chunk in r.iter_content(chunk_size=1024*1024):
                    if chunk: f.write(chunk)
            
            if tmp_path.stat().st_size == 0:
                tmp_path.unlink()
                raise RuntimeError("Empty file received")
            
            tmp_path.replace(fpath)
            return f"Downloaded: {fname}"
            
    except Exception as e:
        raise RuntimeError(f"Network error: {str(e)}")

In [15]:
# Map: KEY -> URL (Paste your URLs here!)
URL_TEMPLATES = {

##################### INDUSTRIAL DESIGNS ###########################################################################################################
    # fct_ip_flows_yearly
    "ID1a": UrlSpec(url_template="https://api.ipstatsdc.deda.prd.web1.wipo.int/api/v1/public/ips-search/downloadCsv?selectedTab=industrial&indicator=51&reportType=15&fromYear={from_year}&toYear={to_year}&ipsOffSelValues=AF,OA,AP,AL,DZ,AD,AO,AG,AR,AM,AU,AT,AZ,BS,BH,BD,BB,BY,BE,BZ,BX,BJ,BT,BO,BQ,BA,BW,BR,BN,BG,BF,BI,CV,KH,CM,CA,CF,TD,CL,CN,HK,MO,CO,KM,CG,CK,CR,CI,HR,CU,CW,CY,CZ,CS,KP,CD,DK,DJ,DM,DO,EC,EG,SV,GQ,ER,EE,SZ,ET,EA,EM,FJ,FI,FR,GA,GM,GE,DD,DE,GH,GR,GD,GT,GG,GN,GW,GY,HT,VA,HN,HU,IS,IN,ID,IR,IQ,IE,IL,IT,JM,JP,JO,KZ,KE,KI,KW,KG,LA,LV,LB,LS,LR,LY,LI,LT,LU,MG,MW,MY,MV,ML,MT,MH,MR,MU,MX,FM,MC,MN,ME,MA,MZ,MM,NA,NR,NP,NL,AN,NZ,NI,NE,NG,NU,MK,NO,OM,PK,PW,PA,PG,PY,PE,PH,PL,PT,QA,KR,MD,RO,RU,RW,KN,LC,VC,WS,SM,ST,SA,SN,RS,SC,SL,SG,SX,SK,SI,SB,SO,ZA,SS,SU,ES,LK,SD,SR,SE,CH,SY,TJ,TH,TL,TG,TO,TT,TN,TR,TM,TV,UG,UA,AE,GB,TZ,US,UY,UZ,VU,VE,VN,YE,YU,ZR,ZM,ZW&ipsOriSelValues=AF,AL,DZ,AD,AO,AG,AR,AM,AU,AT,AZ,BS,BH,BD,BB,BY,BE,BZ,BJ,BT,BO,BQ,BA,BW,BR,BN,BG,BF,BI,CV,KH,CM,CA,CF,TD,CL,CN,HK,MO,CO,KM,CG,CK,CR,CI,HR,CU,CW,CY,CZ,CS,KP,CD,DK,DJ,DM,DO,EC,EG,SV,GQ,ER,EE,SZ,ET,FJ,FI,FR,GA,GM,GE,DD,DE,GH,GR,GD,GT,GN,GW,GY,HT,VA,HN,HU,IS,IN,ID,IR,IQ,IE,IL,IT,JM,JP,JO,KZ,KE,KI,KW,KG,LA,LV,LB,LS,LR,LY,LI,LT,LU,MG,MW,MY,MV,ML,MT,MH,MR,MU,MX,FM,MC,MN,ME,MA,MZ,MM,NA,NR,NP,NL,AN,NZ,NI,NE,NG,NU,MK,NO,OM,PK,PW,PA,PG,PY,PE,PH,PL,PT,QA,KR,MD,RO,RU,RW,KN,LC,VC,WS,SM,ST,SA,SN,RS,SC,SL,SG,SX,SK,SI,SB,SO,ZA,SS,SU,ES,LK,SD,SR,SE,CH,SY,TJ,TH,TL,TG,TO,TT,TN,TR,TM,TV,UG,UA,AE,GB,TZ,US,UY,UZ,VU,VE,VN,YE,YU,ZR,ZM,ZW,**"),
    "ID1b": UrlSpec(url_template="https://api.ipstatsdc.deda.prd.web1.wipo.int/api/v1/public/ips-search/downloadCsv?selectedTab=industrial&indicator=52&reportType=15&fromYear={from_year}&toYear={to_year}&ipsOffSelValues=AF,OA,AP,AL,DZ,AD,AO,AG,AR,AM,AU,AT,AZ,BS,BH,BD,BB,BY,BE,BZ,BX,BJ,BT,BO,BQ,BA,BW,BR,BN,BG,BF,BI,CV,KH,CM,CA,CF,TD,CL,CN,HK,MO,CO,KM,CG,CK,CR,CI,HR,CU,CW,CY,CZ,CS,KP,CD,DK,DJ,DM,DO,EC,EG,SV,GQ,ER,EE,SZ,ET,EA,EM,FJ,FI,FR,GA,GM,GE,DD,DE,GH,GR,GD,GT,GG,GN,GW,GY,HT,VA,HN,HU,IS,IN,ID,IR,IQ,IE,IL,IT,JM,JP,JO,KZ,KE,KI,KW,KG,LA,LV,LB,LS,LR,LY,LI,LT,LU,MG,MW,MY,MV,ML,MT,MH,MR,MU,MX,FM,MC,MN,ME,MA,MZ,MM,NA,NR,NP,NL,AN,NZ,NI,NE,NG,NU,MK,NO,OM,PK,PW,PA,PG,PY,PE,PH,PL,PT,QA,KR,MD,RO,RU,RW,KN,LC,VC,WS,SM,ST,SA,SN,RS,SC,SL,SG,SX,SK,SI,SB,SO,ZA,SS,SU,ES,LK,SD,SR,SE,CH,SY,TJ,TH,TL,TG,TO,TT,TN,TR,TM,TV,UG,UA,AE,GB,TZ,US,UY,UZ,VU,VE,VN,YE,YU,ZR,ZM,ZW&ipsOriSelValues=AF,AL,DZ,AD,AO,AG,AR,AM,AU,AT,AZ,BS,BH,BD,BB,BY,BE,BZ,BJ,BT,BO,BQ,BA,BW,BR,BN,BG,BF,BI,CV,KH,CM,CA,CF,TD,CL,CN,HK,MO,CO,KM,CG,CK,CR,CI,HR,CU,CW,CY,CZ,CS,KP,CD,DK,DJ,DM,DO,EC,EG,SV,GQ,ER,EE,SZ,ET,FJ,FI,FR,GA,GM,GE,DD,DE,GH,GR,GD,GT,GN,GW,GY,HT,VA,HN,HU,IS,IN,ID,IR,IQ,IE,IL,IT,JM,JP,JO,KZ,KE,KI,KW,KG,LA,LV,LB,LS,LR,LY,LI,LT,LU,MG,MW,MY,MV,ML,MT,MH,MR,MU,MX,FM,MC,MN,ME,MA,MZ,MM,NA,NR,NP,NL,AN,NZ,NI,NE,NG,NU,MK,NO,OM,PK,PW,PA,PG,PY,PE,PH,PL,PT,QA,KR,MD,RO,RU,RW,KN,LC,VC,WS,SM,ST,SA,SN,RS,SC,SL,SG,SX,SK,SI,SB,SO,ZA,SS,SU,ES,LK,SD,SR,SE,CH,SY,TJ,TH,TL,TG,TO,TT,TN,TR,TM,TV,UG,UA,AE,GB,TZ,US,UY,UZ,VU,VE,VN,YE,YU,ZR,ZM,ZW,**"),
    "ID2a": UrlSpec(url_template="https://api.ipstatsdc.deda.prd.web1.wipo.int/api/v1/public/ips-search/downloadCsv?selectedTab=industrial&indicator=53&reportType=15&fromYear={from_year}&toYear={to_year}&ipsOffSelValues=AF,OA,AP,AL,DZ,AD,AO,AG,AR,AM,AU,AT,AZ,BS,BH,BD,BB,BY,BE,BZ,BX,BJ,BT,BO,BQ,BA,BW,BR,BN,BG,BF,BI,CV,KH,CM,CA,CF,TD,CL,CN,HK,MO,CO,KM,CG,CK,CR,CI,HR,CU,CW,CY,CZ,CS,KP,CD,DK,DJ,DM,DO,EC,EG,SV,GQ,ER,EE,SZ,ET,EA,EM,FJ,FI,FR,GA,GM,GE,DD,DE,GH,GR,GD,GT,GG,GN,GW,GY,HT,VA,HN,HU,IS,IN,ID,IR,IQ,IE,IL,IT,JM,JP,JO,KZ,KE,KI,KW,KG,LA,LV,LB,LS,LR,LY,LI,LT,LU,MG,MW,MY,MV,ML,MT,MH,MR,MU,MX,FM,MC,MN,ME,MA,MZ,MM,NA,NR,NP,NL,AN,NZ,NI,NE,NG,NU,MK,NO,OM,PK,PW,PA,PG,PY,PE,PH,PL,PT,QA,KR,MD,RO,RU,RW,KN,LC,VC,WS,SM,ST,SA,SN,RS,SC,SL,SG,SX,SK,SI,SB,SO,ZA,SS,SU,ES,LK,SD,SR,SE,CH,SY,TJ,TH,TL,TG,TO,TT,TN,TR,TM,TV,UG,UA,AE,GB,TZ,US,UY,UZ,VU,VE,VN,YE,YU,ZR,ZM,ZW&ipsOriSelValues=AF,AL,DZ,AD,AO,AG,AR,AM,AU,AT,AZ,BS,BH,BD,BB,BY,BE,BZ,BJ,BT,BO,BQ,BA,BW,BR,BN,BG,BF,BI,CV,KH,CM,CA,CF,TD,CL,CN,HK,MO,CO,KM,CG,CK,CR,CI,HR,CU,CW,CY,CZ,CS,KP,CD,DK,DJ,DM,DO,EC,EG,SV,GQ,ER,EE,SZ,ET,FJ,FI,FR,GA,GM,GE,DD,DE,GH,GR,GD,GT,GN,GW,GY,HT,VA,HN,HU,IS,IN,ID,IR,IQ,IE,IL,IT,JM,JP,JO,KZ,KE,KI,KW,KG,LA,LV,LB,LS,LR,LY,LI,LT,LU,MG,MW,MY,MV,ML,MT,MH,MR,MU,MX,FM,MC,MN,ME,MA,MZ,MM,NA,NR,NP,NL,AN,NZ,NI,NE,NG,NU,MK,NO,OM,PK,PW,PA,PG,PY,PE,PH,PL,PT,QA,KR,MD,RO,RU,RW,KN,LC,VC,WS,SM,ST,SA,SN,RS,SC,SL,SG,SX,SK,SI,SB,SO,ZA,SS,SU,ES,LK,SD,SR,SE,CH,SY,TJ,TH,TL,TG,TO,TT,TN,TR,TM,TV,UG,UA,AE,GB,TZ,US,UY,UZ,VU,VE,VN,YE,YU,ZR,ZM,ZW,**"),
    "ID2b": UrlSpec(url_template="https://api.ipstatsdc.deda.prd.web1.wipo.int/api/v1/public/ips-search/downloadCsv?selectedTab=industrial&indicator=54&reportType=15&fromYear={from_year}&toYear={to_year}&ipsOffSelValues=AF,OA,AP,AL,DZ,AD,AO,AG,AR,AM,AU,AT,AZ,BS,BH,BD,BB,BY,BE,BZ,BX,BJ,BT,BO,BQ,BA,BW,BR,BN,BG,BF,BI,CV,KH,CM,CA,CF,TD,CL,CN,HK,MO,CO,KM,CG,CK,CR,CI,HR,CU,CW,CY,CZ,CS,KP,CD,DK,DJ,DM,DO,EC,EG,SV,GQ,ER,EE,SZ,ET,EA,EM,FJ,FI,FR,GA,GM,GE,DD,DE,GH,GR,GD,GT,GG,GN,GW,GY,HT,VA,HN,HU,IS,IN,ID,IR,IQ,IE,IL,IT,JM,JP,JO,KZ,KE,KI,KW,KG,LA,LV,LB,LS,LR,LY,LI,LT,LU,MG,MW,MY,MV,ML,MT,MH,MR,MU,MX,FM,MC,MN,ME,MA,MZ,MM,NA,NR,NP,NL,AN,NZ,NI,NE,NG,NU,MK,NO,OM,PK,PW,PA,PG,PY,PE,PH,PL,PT,QA,KR,MD,RO,RU,RW,KN,LC,VC,WS,SM,ST,SA,SN,RS,SC,SL,SG,SX,SK,SI,SB,SO,ZA,SS,SU,ES,LK,SD,SR,SE,CH,SY,TJ,TH,TL,TG,TO,TT,TN,TR,TM,TV,UG,UA,AE,GB,TZ,US,UY,UZ,VU,VE,VN,YE,YU,ZR,ZM,ZW&ipsOriSelValues=AF,AL,DZ,AD,AO,AG,AR,AM,AU,AT,AZ,BS,BH,BD,BB,BY,BE,BZ,BJ,BT,BO,BQ,BA,BW,BR,BN,BG,BF,BI,CV,KH,CM,CA,CF,TD,CL,CN,HK,MO,CO,KM,CG,CK,CR,CI,HR,CU,CW,CY,CZ,CS,KP,CD,DK,DJ,DM,DO,EC,EG,SV,GQ,ER,EE,SZ,ET,FJ,FI,FR,GA,GM,GE,DD,DE,GH,GR,GD,GT,GN,GW,GY,HT,VA,HN,HU,IS,IN,ID,IR,IQ,IE,IL,IT,JM,JP,JO,KZ,KE,KI,KW,KG,LA,LV,LB,LS,LR,LY,LI,LT,LU,MG,MW,MY,MV,ML,MT,MH,MR,MU,MX,FM,MC,MN,ME,MA,MZ,MM,NA,NR,NP,NL,AN,NZ,NI,NE,NG,NU,MK,NO,OM,PK,PW,PA,PG,PY,PE,PH,PL,PT,QA,KR,MD,RO,RU,RW,KN,LC,VC,WS,SM,ST,SA,SN,RS,SC,SL,SG,SX,SK,SI,SB,SO,ZA,SS,SU,ES,LK,SD,SR,SE,CH,SY,TJ,TH,TL,TG,TO,TT,TN,TR,TM,TV,UG,UA,AE,GB,TZ,US,UY,UZ,VU,VE,VN,YE,YU,ZR,ZM,ZW,**"),
    "ID7":  UrlSpec(url_template="https://api.ipstatsdc.deda.prd.web1.wipo.int/api/v1/public/ips-search/downloadCsv?selectedTab=industrial&indicator=63&reportType=11&fromYear={from_year}&toYear={to_year}&ipsOffSelValues=AF,OA,AP,AL,DZ,AD,AO,AG,AR,AM,AU,AT,AZ,BS,BH,BD,BB,BY,BE,BZ,BX,BJ,BT,BO,BQ,BA,BW,BR,BN,BG,BF,BI,CV,KH,CM,CA,CF,TD,CL,CN,HK,MO,CO,KM,CG,CK,CR,CI,HR,CU,CW,CY,CZ,CS,KP,CD,DK,DJ,DM,DO,EC,EG,SV,GQ,ER,EE,SZ,ET,EA,EM,FJ,FI,FR,GA,GM,GE,DD,DE,GH,GR,GD,GT,GG,GN,GW,GY,HT,VA,HN,HU,IS,IN,ID,IR,IQ,IE,IL,IT,JM,JP,JO,KZ,KE,KI,KW,KG,LA,LV,LB,LS,LR,LY,LI,LT,LU,MG,MW,MY,MV,ML,MT,MH,MR,MU,MX,FM,MC,MN,ME,MA,MZ,MM,NA,NR,NP,NL,AN,NZ,NI,NE,NG,NU,MK,NO,OM,PK,PW,PA,PG,PY,PE,PH,PL,PT,QA,KR,MD,RO,RU,RW,KN,LC,VC,WS,SM,ST,SA,SN,RS,SC,SL,SG,SX,SK,SI,SB,SO,ZA,SS,SU,ES,LK,SD,SR,SE,CH,SY,TJ,TH,TL,TG,TO,TT,TN,TR,TM,TV,UG,UA,AE,GB,TZ,US,UY,UZ,VU,VE,VN,YE,YU,ZR,ZM,ZW"),


    #fct_ip_flows_by_class_yearly
    "ID3a": UrlSpec(url_template="https://api.ipstatsdc.deda.prd.web1.wipo.int/api/v1/public/ips-search/downloadCsv?selectedTab=industrial&indicator=55&reportType=15&fromYear={from_year}&toYear={to_year}&ipsOffSelValues=AF,OA,AP,AL,DZ,AD,AO,AG,AR,AM,AU,AT,AZ,BS,BH,BD,BB,BY,BE,BZ,BX,BJ,BT,BO,BQ,BA,BW,BR,BN,BG,BF,BI,CV,KH,CM,CA,CF,TD,CL,CN,HK,MO,CO,KM,CG,CK,CR,CI,HR,CU,CW,CY,CZ,CS,KP,CD,DK,DJ,DM,DO,EC,EG,SV,GQ,ER,EE,SZ,ET,EA,EM,FJ,FI,FR,GA,GM,GE,DD,DE,GH,GR,GD,GT,GG,GN,GW,GY,HT,VA,HN,HU,IS,IN,ID,IR,IQ,IE,IL,IT,JM,JP,JO,KZ,KE,KI,KW,KG,LA,LV,LB,LS,LR,LY,LI,LT,LU,MG,MW,MY,MV,ML,MT,MH,MR,MU,MX,FM,MC,MN,ME,MA,MZ,MM,NA,NR,NP,NL,AN,NZ,NI,NE,NG,NU,MK,NO,OM,PK,PW,PA,PG,PY,PE,PH,PL,PT,QA,KR,MD,RO,RU,RW,KN,LC,VC,WS,SM,ST,SA,SN,RS,SC,SL,SG,SX,SK,SI,SB,SO,ZA,SS,SU,ES,LK,SD,SR,SE,CH,SY,TJ,TH,TL,TG,TO,TT,TN,TR,TM,TV,UG,UA,AE,GB,TZ,US,UY,UZ,VU,VE,VN,YE,YU,ZR,ZM,ZW&ipsOriSelValues=AF,AL,DZ,AD,AO,AG,AR,AM,AU,AT,AZ,BS,BH,BD,BB,BY,BE,BZ,BJ,BT,BO,BQ,BA,BW,BR,BN,BG,BF,BI,CV,KH,CM,CA,CF,TD,CL,CN,HK,MO,CO,KM,CG,CK,CR,CI,HR,CU,CW,CY,CZ,CS,KP,CD,DK,DJ,DM,DO,EC,EG,SV,GQ,ER,EE,SZ,ET,FJ,FI,FR,GA,GM,GE,DD,DE,GH,GR,GD,GT,GN,GW,GY,HT,VA,HN,HU,IS,IN,ID,IR,IQ,IE,IL,IT,JM,JP,JO,KZ,KE,KI,KW,KG,LA,LV,LB,LS,LR,LY,LI,LT,LU,MG,MW,MY,MV,ML,MT,MH,MR,MU,MX,FM,MC,MN,ME,MA,MZ,MM,NA,NR,NP,NL,AN,NZ,NI,NE,NG,NU,MK,NO,OM,PK,PW,PA,PG,PY,PE,PH,PL,PT,QA,KR,MD,RO,RU,RW,KN,LC,VC,WS,SM,ST,SA,SN,RS,SC,SL,SG,SX,SK,SI,SB,SO,ZA,SS,SU,ES,LK,SD,SR,SE,CH,SY,TJ,TH,TL,TG,TO,TT,TN,TR,TM,TV,UG,UA,AE,GB,TZ,US,UY,UZ,VU,VE,VN,YE,YU,ZR,ZM,ZW,**&ipsTechSelValues=200,201,202,203,204,205,206,207,208,209,210,211,212,213,214,215,216,217,218,219,220,221,222,223,224,225,226,227,228,229,230,231,232"),
    "ID3b": UrlSpec(url_template="https://api.ipstatsdc.deda.prd.web1.wipo.int/api/v1/public/ips-search/downloadCsv?selectedTab=industrial&indicator=56&reportType=15&fromYear={from_year}&toYear={to_year}&ipsOffSelValues=AF,OA,AP,AL,DZ,AD,AO,AG,AR,AM,AU,AT,AZ,BS,BH,BD,BB,BY,BE,BZ,BX,BJ,BT,BO,BQ,BA,BW,BR,BN,BG,BF,BI,CV,KH,CM,CA,CF,TD,CL,CN,HK,MO,CO,KM,CG,CK,CR,CI,HR,CU,CW,CY,CZ,CS,KP,CD,DK,DJ,DM,DO,EC,EG,SV,GQ,ER,EE,SZ,ET,EA,EM,FJ,FI,FR,GA,GM,GE,DD,DE,GH,GR,GD,GT,GG,GN,GW,GY,HT,VA,HN,HU,IS,IN,ID,IR,IQ,IE,IL,IT,JM,JP,JO,KZ,KE,KI,KW,KG,LA,LV,LB,LS,LR,LY,LI,LT,LU,MG,MW,MY,MV,ML,MT,MH,MR,MU,MX,FM,MC,MN,ME,MA,MZ,MM,NA,NR,NP,NL,AN,NZ,NI,NE,NG,NU,MK,NO,OM,PK,PW,PA,PG,PY,PE,PH,PL,PT,QA,KR,MD,RO,RU,RW,KN,LC,VC,WS,SM,ST,SA,SN,RS,SC,SL,SG,SX,SK,SI,SB,SO,ZA,SS,SU,ES,LK,SD,SR,SE,CH,SY,TJ,TH,TL,TG,TO,TT,TN,TR,TM,TV,UG,UA,AE,GB,TZ,US,UY,UZ,VU,VE,VN,YE,YU,ZR,ZM,ZW&ipsOriSelValues=AF,AL,DZ,AD,AO,AG,AR,AM,AU,AT,AZ,BS,BH,BD,BB,BY,BE,BZ,BJ,BT,BO,BQ,BA,BW,BR,BN,BG,BF,BI,CV,KH,CM,CA,CF,TD,CL,CN,HK,MO,CO,KM,CG,CK,CR,CI,HR,CU,CW,CY,CZ,CS,KP,CD,DK,DJ,DM,DO,EC,EG,SV,GQ,ER,EE,SZ,ET,FJ,FI,FR,GA,GM,GE,DD,DE,GH,GR,GD,GT,GN,GW,GY,HT,VA,HN,HU,IS,IN,ID,IR,IQ,IE,IL,IT,JM,JP,JO,KZ,KE,KI,KW,KG,LA,LV,LB,LS,LR,LY,LI,LT,LU,MG,MW,MY,MV,ML,MT,MH,MR,MU,MX,FM,MC,MN,ME,MA,MZ,MM,NA,NR,NP,NL,AN,NZ,NI,NE,NG,NU,MK,NO,OM,PK,PW,PA,PG,PY,PE,PH,PL,PT,QA,KR,MD,RO,RU,RW,KN,LC,VC,WS,SM,ST,SA,SN,RS,SC,SL,SG,SX,SK,SI,SB,SO,ZA,SS,SU,ES,LK,SD,SR,SE,CH,SY,TJ,TH,TL,TG,TO,TT,TN,TR,TM,TV,UG,UA,AE,GB,TZ,US,UY,UZ,VU,VE,VN,YE,YU,ZR,ZM,ZW,**&ipsTechSelValues=200,201,202,203,204,205,206,207,208,209,210,211,212,213,214,215,216,217,218,219,220,221,222,223,224,225,226,227,228,229,230,231,232"),
    "ID4a": UrlSpec(url_template="https://api.ipstatsdc.deda.prd.web1.wipo.int/api/v1/public/ips-search/downloadCsv?selectedTab=industrial&indicator=57&reportType=15&fromYear={from_year}&toYear={to_year}&ipsOffSelValues=AF,OA,AP,AL,DZ,AD,AO,AG,AR,AM,AU,AT,AZ,BS,BH,BD,BB,BY,BE,BZ,BX,BJ,BT,BO,BQ,BA,BW,BR,BN,BG,BF,BI,CV,KH,CM,CA,CF,TD,CL,CN,HK,MO,CO,KM,CG,CK,CR,CI,HR,CU,CW,CY,CZ,CS,KP,CD,DK,DJ,DM,DO,EC,EG,SV,GQ,ER,EE,SZ,ET,EA,EM,FJ,FI,FR,GA,GM,GE,DD,DE,GH,GR,GD,GT,GG,GN,GW,GY,HT,VA,HN,HU,IS,IN,ID,IR,IQ,IE,IL,IT,JM,JP,JO,KZ,KE,KI,KW,KG,LA,LV,LB,LS,LR,LY,LI,LT,LU,MG,MW,MY,MV,ML,MT,MH,MR,MU,MX,FM,MC,MN,ME,MA,MZ,MM,NA,NR,NP,NL,AN,NZ,NI,NE,NG,NU,MK,NO,OM,PK,PW,PA,PG,PY,PE,PH,PL,PT,QA,KR,MD,RO,RU,RW,KN,LC,VC,WS,SM,ST,SA,SN,RS,SC,SL,SG,SX,SK,SI,SB,SO,ZA,SS,SU,ES,LK,SD,SR,SE,CH,SY,TJ,TH,TL,TG,TO,TT,TN,TR,TM,TV,UG,UA,AE,GB,TZ,US,UY,UZ,VU,VE,VN,YE,YU,ZR,ZM,ZW&ipsOriSelValues=AF,AL,DZ,AD,AO,AG,AR,AM,AU,AT,AZ,BS,BH,BD,BB,BY,BE,BZ,BJ,BT,BO,BQ,BA,BW,BR,BN,BG,BF,BI,CV,KH,CM,CA,CF,TD,CL,CN,HK,MO,CO,KM,CG,CK,CR,CI,HR,CU,CW,CY,CZ,CS,KP,CD,DK,DJ,DM,DO,EC,EG,SV,GQ,ER,EE,SZ,ET,FJ,FI,FR,GA,GM,GE,DD,DE,GH,GR,GD,GT,GN,GW,GY,HT,VA,HN,HU,IS,IN,ID,IR,IQ,IE,IL,IT,JM,JP,JO,KZ,KE,KI,KW,KG,LA,LV,LB,LS,LR,LY,LI,LT,LU,MG,MW,MY,MV,ML,MT,MH,MR,MU,MX,FM,MC,MN,ME,MA,MZ,MM,NA,NR,NP,NL,AN,NZ,NI,NE,NG,NU,MK,NO,OM,PK,PW,PA,PG,PY,PE,PH,PL,PT,QA,KR,MD,RO,RU,RW,KN,LC,VC,WS,SM,ST,SA,SN,RS,SC,SL,SG,SX,SK,SI,SB,SO,ZA,SS,SU,ES,LK,SD,SR,SE,CH,SY,TJ,TH,TL,TG,TO,TT,TN,TR,TM,TV,UG,UA,AE,GB,TZ,US,UY,UZ,VU,VE,VN,YE,YU,ZR,ZM,ZW,**&ipsTechSelValues=200,201,202,203,204,205,206,207,208,209,210,211,212,213,214,215,216,217,218,219,220,221,222,223,224,225,226,227,228,229,230,231,232"),
    "ID4b": UrlSpec(url_template="https://api.ipstatsdc.deda.prd.web1.wipo.int/api/v1/public/ips-search/downloadCsv?selectedTab=industrial&indicator=58&reportType=15&fromYear={from_year}&toYear={to_year}&ipsOffSelValues=AF,OA,AP,AL,DZ,AD,AO,AG,AR,AM,AU,AT,AZ,BS,BH,BD,BB,BY,BE,BZ,BX,BJ,BT,BO,BQ,BA,BW,BR,BN,BG,BF,BI,CV,KH,CM,CA,CF,TD,CL,CN,HK,MO,CO,KM,CG,CK,CR,CI,HR,CU,CW,CY,CZ,CS,KP,CD,DK,DJ,DM,DO,EC,EG,SV,GQ,ER,EE,SZ,ET,EA,EM,FJ,FI,FR,GA,GM,GE,DD,DE,GH,GR,GD,GT,GG,GN,GW,GY,HT,VA,HN,HU,IS,IN,ID,IR,IQ,IE,IL,IT,JM,JP,JO,KZ,KE,KI,KW,KG,LA,LV,LB,LS,LR,LY,LI,LT,LU,MG,MW,MY,MV,ML,MT,MH,MR,MU,MX,FM,MC,MN,ME,MA,MZ,MM,NA,NR,NP,NL,AN,NZ,NI,NE,NG,NU,MK,NO,OM,PK,PW,PA,PG,PY,PE,PH,PL,PT,QA,KR,MD,RO,RU,RW,KN,LC,VC,WS,SM,ST,SA,SN,RS,SC,SL,SG,SX,SK,SI,SB,SO,ZA,SS,SU,ES,LK,SD,SR,SE,CH,SY,TJ,TH,TL,TG,TO,TT,TN,TR,TM,TV,UG,UA,AE,GB,TZ,US,UY,UZ,VU,VE,VN,YE,YU,ZR,ZM,ZW&ipsOriSelValues=AF,AL,DZ,AD,AO,AG,AR,AM,AU,AT,AZ,BS,BH,BD,BB,BY,BE,BZ,BJ,BT,BO,BQ,BA,BW,BR,BN,BG,BF,BI,CV,KH,CM,CA,CF,TD,CL,CN,HK,MO,CO,KM,CG,CK,CR,CI,HR,CU,CW,CY,CZ,CS,KP,CD,DK,DJ,DM,DO,EC,EG,SV,GQ,ER,EE,SZ,ET,FJ,FI,FR,GA,GM,GE,DD,DE,GH,GR,GD,GT,GN,GW,GY,HT,VA,HN,HU,IS,IN,ID,IR,IQ,IE,IL,IT,JM,JP,JO,KZ,KE,KI,KW,KG,LA,LV,LB,LS,LR,LY,LI,LT,LU,MG,MW,MY,MV,ML,MT,MH,MR,MU,MX,FM,MC,MN,ME,MA,MZ,MM,NA,NR,NP,NL,AN,NZ,NI,NE,NG,NU,MK,NO,OM,PK,PW,PA,PG,PY,PE,PH,PL,PT,QA,KR,MD,RO,RU,RW,KN,LC,VC,WS,SM,ST,SA,SN,RS,SC,SL,SG,SX,SK,SI,SB,SO,ZA,SS,SU,ES,LK,SD,SR,SE,CH,SY,TJ,TH,TL,TG,TO,TT,TN,TR,TM,TV,UG,UA,AE,GB,TZ,US,UY,UZ,VU,VE,VN,YE,YU,ZR,ZM,ZW,**&ipsTechSelValues=200,201,202,203,204,205,206,207,208,209,210,211,212,213,214,215,216,217,218,219,220,221,222,223,224,225,226,227,228,229,230,231,232"),

    #fct_ip_flows_by_residency
    "ID1a_origin": UrlSpec(url_template="https://api.ipstatsdc.deda.prd.web1.wipo.int/api/v1/public/ips-search/downloadCsv?selectedTab=industrial&indicator=51&reportType=13&fromYear={from_year}&toYear={to_year}&ipsOriSelValues=AF,AL,DZ,AD,AO,AG,AR,AM,AU,AT,AZ,BS,BH,BD,BB,BY,BE,BZ,BJ,BT,BO,BQ,BA,BW,BR,BN,BG,BF,BI,CV,KH,CM,CA,CF,TD,CL,CN,HK,MO,CO,KM,CG,CK,CR,CI,HR,CU,CW,CY,CZ,CS,KP,CD,DK,DJ,DM,DO,EC,EG,SV,GQ,ER,EE,SZ,ET,FJ,FI,FR,GA,GM,GE,DD,DE,GH,GR,GD,GT,GN,GW,GY,HT,VA,HN,HU,IS,IN,ID,IR,IQ,IE,IL,IT,JM,JP,JO,KZ,KE,KI,KW,KG,LA,LV,LB,LS,LR,LY,LI,LT,LU,MG,MW,MY,MV,ML,MT,MH,MR,MU,MX,FM,MC,MN,ME,MA,MZ,MM,NA,NR,NP,NL,AN,NZ,NI,NE,NG,NU,MK,NO,OM,PK,PW,PA,PG,PY,PE,PH,PL,PT,QA,KR,MD,RO,RU,RW,KN,LC,VC,WS,SM,ST,SA,SN,RS,SC,SL,SG,SX,SK,SI,SB,SO,ZA,SS,SU,ES,LK,SD,SR,SE,CH,SY,TJ,TH,TL,TG,TO,TT,TN,TR,TM,TV,UG,UA,AE,GB,TZ,US,UY,UZ,VU,VE,VN,YE,YU,ZR,ZM,ZW&ipsTechSelValues=910,911,912,913,914"),
    "ID1b_origin": UrlSpec(url_template="https://api.ipstatsdc.deda.prd.web1.wipo.int/api/v1/public/ips-search/downloadCsv?selectedTab=industrial&indicator=52&reportType=13&fromYear={from_year}&toYear={to_year}&ipsOriSelValues=AF,AL,DZ,AD,AO,AG,AR,AM,AU,AT,AZ,BS,BH,BD,BB,BY,BE,BZ,BJ,BT,BO,BQ,BA,BW,BR,BN,BG,BF,BI,CV,KH,CM,CA,CF,TD,CL,CN,HK,MO,CO,KM,CG,CK,CR,CI,HR,CU,CW,CY,CZ,CS,KP,CD,DK,DJ,DM,DO,EC,EG,SV,GQ,ER,EE,SZ,ET,FJ,FI,FR,GA,GM,GE,DD,DE,GH,GR,GD,GT,GN,GW,GY,HT,VA,HN,HU,IS,IN,ID,IR,IQ,IE,IL,IT,JM,JP,JO,KZ,KE,KI,KW,KG,LA,LV,LB,LS,LR,LY,LI,LT,LU,MG,MW,MY,MV,ML,MT,MH,MR,MU,MX,FM,MC,MN,ME,MA,MZ,MM,NA,NR,NP,NL,AN,NZ,NI,NE,NG,NU,MK,NO,OM,PK,PW,PA,PG,PY,PE,PH,PL,PT,QA,KR,MD,RO,RU,RW,KN,LC,VC,WS,SM,ST,SA,SN,RS,SC,SL,SG,SX,SK,SI,SB,SO,ZA,SS,SU,ES,LK,SD,SR,SE,CH,SY,TJ,TH,TL,TG,TO,TT,TN,TR,TM,TV,UG,UA,AE,GB,TZ,US,UY,UZ,VU,VE,VN,YE,YU,ZR,ZM,ZW&ipsTechSelValues=910,911,912,913,914"),
    "ID1a_office": UrlSpec(url_template="https://api.ipstatsdc.deda.prd.web1.wipo.int/api/v1/public/ips-search/downloadCsv?selectedTab=industrial&indicator=51&reportType=11&fromYear={from_year}&toYear={to_year}&ipsOffSelValues=AF,OA,AP,AL,DZ,AD,AO,AG,AR,AM,AU,AT,AZ,BS,BH,BD,BB,BY,BE,BZ,BX,BJ,BT,BO,BQ,BA,BW,BR,BN,BG,BF,BI,CV,KH,CM,CA,CF,TD,CL,CN,HK,MO,CO,KM,CG,CK,CR,CI,HR,CU,CW,CY,CZ,CS,KP,CD,DK,DJ,DM,DO,EC,EG,SV,GQ,ER,EE,SZ,ET,EA,EM,FJ,FI,FR,GA,GM,GE,DD,DE,GH,GR,GD,GT,GG,GN,GW,GY,HT,VA,HN,HU,IS,IN,ID,IR,IQ,IE,IL,IT,JM,JP,JO,KZ,KE,KI,KW,KG,LA,LV,LB,LS,LR,LY,LI,LT,LU,MG,MW,MY,MV,ML,MT,MH,MR,MU,MX,FM,MC,MN,ME,MA,MZ,MM,NA,NR,NP,NL,AN,NZ,NI,NE,NG,NU,MK,NO,OM,PK,PW,PA,PG,PY,PE,PH,PL,PT,QA,KR,MD,RO,RU,RW,KN,LC,VC,WS,SM,ST,SA,SN,RS,SC,SL,SG,SX,SK,SI,SB,SO,ZA,SS,SU,ES,LK,SD,SR,SE,CH,SY,TJ,TH,TL,TG,TO,TT,TN,TR,TM,TV,UG,UA,AE,GB,TZ,US,UY,UZ,VU,VE,VN,YE,YU,ZR,ZM,ZW,WD,R1,R2,R3,R4,R5,R6&ipsTechSelValues=900,901,902"),
    "ID1b_office": UrlSpec(url_template="https://api.ipstatsdc.deda.prd.web1.wipo.int/api/v1/public/ips-search/downloadCsv?selectedTab=industrial&indicator=52&reportType=11&fromYear={from_year}&toYear={to_year}&ipsOffSelValues=AF,OA,AP,AL,DZ,AD,AO,AG,AR,AM,AU,AT,AZ,BS,BH,BD,BB,BY,BE,BZ,BX,BJ,BT,BO,BQ,BA,BW,BR,BN,BG,BF,BI,CV,KH,CM,CA,CF,TD,CL,CN,HK,MO,CO,KM,CG,CK,CR,CI,HR,CU,CW,CY,CZ,CS,KP,CD,DK,DJ,DM,DO,EC,EG,SV,GQ,ER,EE,SZ,ET,EA,EM,FJ,FI,FR,GA,GM,GE,DD,DE,GH,GR,GD,GT,GG,GN,GW,GY,HT,VA,HN,HU,IS,IN,ID,IR,IQ,IE,IL,IT,JM,JP,JO,KZ,KE,KI,KW,KG,LA,LV,LB,LS,LR,LY,LI,LT,LU,MG,MW,MY,MV,ML,MT,MH,MR,MU,MX,FM,MC,MN,ME,MA,MZ,MM,NA,NR,NP,NL,AN,NZ,NI,NE,NG,NU,MK,NO,OM,PK,PW,PA,PG,PY,PE,PH,PL,PT,QA,KR,MD,RO,RU,RW,KN,LC,VC,WS,SM,ST,SA,SN,RS,SC,SL,SG,SX,SK,SI,SB,SO,ZA,SS,SU,ES,LK,SD,SR,SE,CH,SY,TJ,TH,TL,TG,TO,TT,TN,TR,TM,TV,UG,UA,AE,GB,TZ,US,UY,UZ,VU,VE,VN,YE,YU,ZR,ZM,ZW,WD,R1,R2,R3,R4,R5,R6&ipsTechSelValues=900,901,902"),

    "ID2a_origin": UrlSpec(url_template="https://api.ipstatsdc.deda.prd.web1.wipo.int/api/v1/public/ips-search/downloadCsv?selectedTab=industrial&indicator=53&reportType=13&fromYear={from_year}&toYear={to_year}&ipsOriSelValues=AF,AL,DZ,AD,AO,AG,AR,AM,AU,AT,AZ,BS,BH,BD,BB,BY,BE,BZ,BJ,BT,BO,BQ,BA,BW,BR,BN,BG,BF,BI,CV,KH,CM,CA,CF,TD,CL,CN,HK,MO,CO,KM,CG,CK,CR,CI,HR,CU,CW,CY,CZ,CS,KP,CD,DK,DJ,DM,DO,EC,EG,SV,GQ,ER,EE,SZ,ET,FJ,FI,FR,GA,GM,GE,DD,DE,GH,GR,GD,GT,GN,GW,GY,HT,VA,HN,HU,IS,IN,ID,IR,IQ,IE,IL,IT,JM,JP,JO,KZ,KE,KI,KW,KG,LA,LV,LB,LS,LR,LY,LI,LT,LU,MG,MW,MY,MV,ML,MT,MH,MR,MU,MX,FM,MC,MN,ME,MA,MZ,MM,NA,NR,NP,NL,AN,NZ,NI,NE,NG,NU,MK,NO,OM,PK,PW,PA,PG,PY,PE,PH,PL,PT,QA,KR,MD,RO,RU,RW,KN,LC,VC,WS,SM,ST,SA,SN,RS,SC,SL,SG,SX,SK,SI,SB,SO,ZA,SS,SU,ES,LK,SD,SR,SE,CH,SY,TJ,TH,TL,TG,TO,TT,TN,TR,TM,TV,UG,UA,AE,GB,TZ,US,UY,UZ,VU,VE,VN,YE,YU,ZR,ZM,ZW&ipsTechSelValues=910,911,912,913,914"),
    "ID2b_origin": UrlSpec(url_template="https://api.ipstatsdc.deda.prd.web1.wipo.int/api/v1/public/ips-search/downloadCsv?selectedTab=industrial&indicator=54&reportType=13&fromYear={from_year}&toYear={to_year}&ipsOriSelValues=AF,AL,DZ,AD,AO,AG,AR,AM,AU,AT,AZ,BS,BH,BD,BB,BY,BE,BZ,BJ,BT,BO,BQ,BA,BW,BR,BN,BG,BF,BI,CV,KH,CM,CA,CF,TD,CL,CN,HK,MO,CO,KM,CG,CK,CR,CI,HR,CU,CW,CY,CZ,CS,KP,CD,DK,DJ,DM,DO,EC,EG,SV,GQ,ER,EE,SZ,ET,FJ,FI,FR,GA,GM,GE,DD,DE,GH,GR,GD,GT,GN,GW,GY,HT,VA,HN,HU,IS,IN,ID,IR,IQ,IE,IL,IT,JM,JP,JO,KZ,KE,KI,KW,KG,LA,LV,LB,LS,LR,LY,LI,LT,LU,MG,MW,MY,MV,ML,MT,MH,MR,MU,MX,FM,MC,MN,ME,MA,MZ,MM,NA,NR,NP,NL,AN,NZ,NI,NE,NG,NU,MK,NO,OM,PK,PW,PA,PG,PY,PE,PH,PL,PT,QA,KR,MD,RO,RU,RW,KN,LC,VC,WS,SM,ST,SA,SN,RS,SC,SL,SG,SX,SK,SI,SB,SO,ZA,SS,SU,ES,LK,SD,SR,SE,CH,SY,TJ,TH,TL,TG,TO,TT,TN,TR,TM,TV,UG,UA,AE,GB,TZ,US,UY,UZ,VU,VE,VN,YE,YU,ZR,ZM,ZW&ipsTechSelValues=910,911,912,913,914"),
    "ID2a_office": UrlSpec(url_template="https://api.ipstatsdc.deda.prd.web1.wipo.int/api/v1/public/ips-search/downloadCsv?selectedTab=industrial&indicator=53&reportType=11&fromYear={from_year}&toYear={to_year}&ipsOffSelValues=AF,OA,AP,AL,DZ,AD,AO,AG,AR,AM,AU,AT,AZ,BS,BH,BD,BB,BY,BE,BZ,BX,BJ,BT,BO,BQ,BA,BW,BR,BN,BG,BF,BI,CV,KH,CM,CA,CF,TD,CL,CN,HK,MO,CO,KM,CG,CK,CR,CI,HR,CU,CW,CY,CZ,CS,KP,CD,DK,DJ,DM,DO,EC,EG,SV,GQ,ER,EE,SZ,ET,EA,EM,FJ,FI,FR,GA,GM,GE,DD,DE,GH,GR,GD,GT,GG,GN,GW,GY,HT,VA,HN,HU,IS,IN,ID,IR,IQ,IE,IL,IT,JM,JP,JO,KZ,KE,KI,KW,KG,LA,LV,LB,LS,LR,LY,LI,LT,LU,MG,MW,MY,MV,ML,MT,MH,MR,MU,MX,FM,MC,MN,ME,MA,MZ,MM,NA,NR,NP,NL,AN,NZ,NI,NE,NG,NU,MK,NO,OM,PK,PW,PA,PG,PY,PE,PH,PL,PT,QA,KR,MD,RO,RU,RW,KN,LC,VC,WS,SM,ST,SA,SN,RS,SC,SL,SG,SX,SK,SI,SB,SO,ZA,SS,SU,ES,LK,SD,SR,SE,CH,SY,TJ,TH,TL,TG,TO,TT,TN,TR,TM,TV,UG,UA,AE,GB,TZ,US,UY,UZ,VU,VE,VN,YE,YU,ZR,ZM,ZW,WD,R1,R2,R3,R4,R5,R6&ipsTechSelValues=900,901,902"),
    "ID2b_office": UrlSpec(url_template="https://api.ipstatsdc.deda.prd.web1.wipo.int/api/v1/public/ips-search/downloadCsv?selectedTab=industrial&indicator=54&reportType=11&fromYear={from_year}&toYear={to_year}&ipsOffSelValues=AF,OA,AP,AL,DZ,AD,AO,AG,AR,AM,AU,AT,AZ,BS,BH,BD,BB,BY,BE,BZ,BX,BJ,BT,BO,BQ,BA,BW,BR,BN,BG,BF,BI,CV,KH,CM,CA,CF,TD,CL,CN,HK,MO,CO,KM,CG,CK,CR,CI,HR,CU,CW,CY,CZ,CS,KP,CD,DK,DJ,DM,DO,EC,EG,SV,GQ,ER,EE,SZ,ET,EA,EM,FJ,FI,FR,GA,GM,GE,DD,DE,GH,GR,GD,GT,GG,GN,GW,GY,HT,VA,HN,HU,IS,IN,ID,IR,IQ,IE,IL,IT,JM,JP,JO,KZ,KE,KI,KW,KG,LA,LV,LB,LS,LR,LY,LI,LT,LU,MG,MW,MY,MV,ML,MT,MH,MR,MU,MX,FM,MC,MN,ME,MA,MZ,MM,NA,NR,NP,NL,AN,NZ,NI,NE,NG,NU,MK,NO,OM,PK,PW,PA,PG,PY,PE,PH,PL,PT,QA,KR,MD,RO,RU,RW,KN,LC,VC,WS,SM,ST,SA,SN,RS,SC,SL,SG,SX,SK,SI,SB,SO,ZA,SS,SU,ES,LK,SD,SR,SE,CH,SY,TJ,TH,TL,TG,TO,TT,TN,TR,TM,TV,UG,UA,AE,GB,TZ,US,UY,UZ,VU,VE,VN,YE,YU,ZR,ZM,ZW,WD,R1,R2,R3,R4,R5,R6&ipsTechSelValues=900,901,902"),
    






##################### PATENTS ###########################################################################################################
    # fct_ip_flows_yearly
    "PA1a": UrlSpec(url_template="https://api.ipstatsdc.deda.prd.web1.wipo.int/api/v1/public/ips-search/downloadCsv?selectedTab=patent&indicator=11&reportType=15&fromYear={from_year}&toYear={to_year}&ipsOffSelValues=AF,OA,AP,AL,DZ,AD,AO,AG,AR,AM,AU,AT,AZ,BS,BH,BD,BB,BY,BE,BZ,BJ,BT,BO,BQ,BA,BW,BR,BN,BG,BF,BI,CV,KH,CM,CA,CF,TD,CL,CN,HK,MO,CO,KM,CG,CK,CR,CI,HR,CU,CW,CY,CZ,CS,KP,CD,DK,DJ,DM,DO,EC,EG,SV,GQ,ER,EE,SZ,ET,EA,EP,FJ,FI,FR,GA,GM,GE,DD,DE,GH,GR,GD,GT,GG,GN,GW,GY,HT,VA,HN,HU,IS,IN,ID,IR,IQ,IE,IL,IT,JM,JP,JO,KZ,KE,KI,KW,KG,LA,LV,LB,LS,LR,LY,LI,LT,LU,MG,MW,MY,MV,ML,MT,MH,MR,MU,MX,FM,MC,MN,ME,MA,MZ,MM,NA,NR,NP,NL,AN,NZ,NI,NE,NG,NU,MK,NO,OM,PK,PW,PA,PG,PY,GC,PE,PH,PL,PT,QA,KR,MD,RO,RU,RW,KN,LC,VC,WS,SM,ST,SA,SN,RS,SC,SL,SG,SX,SK,SI,SB,SO,ZA,SS,SU,ES,LK,SD,SR,SE,CH,SY,TJ,TH,TL,TG,TO,TT,TN,TR,TM,TV,UG,UA,AE,GB,TZ,US,UY,UZ,VU,VE,VN,YE,YU,ZR,ZM,ZW&ipsOriSelValues=AF,AL,DZ,AD,AO,AG,AR,AM,AU,AT,AZ,BS,BH,BD,BB,BY,BE,BZ,BJ,BT,BO,BQ,BA,BW,BR,BN,BG,BF,BI,CV,KH,CM,CA,CF,TD,CL,CN,HK,MO,CO,KM,CG,CK,CR,CI,HR,CU,CW,CY,CZ,CS,KP,CD,DK,DJ,DM,DO,EC,EG,SV,GQ,ER,EE,SZ,ET,FJ,FI,FR,GA,GM,GE,DD,DE,GH,GR,GD,GT,GN,GW,GY,HT,VA,HN,HU,IS,IN,ID,IR,IQ,IE,IL,IT,JM,JP,JO,KZ,KE,KI,KW,KG,LA,LV,LB,LS,LR,LY,LI,LT,LU,MG,MW,MY,MV,ML,MT,MH,MR,MU,MX,FM,MC,MN,ME,MA,MZ,MM,NA,NR,NP,NL,AN,NZ,NI,NE,NG,NU,MK,NO,OM,PK,PW,PA,PG,PY,PE,PH,PL,PT,QA,KR,MD,RO,RU,RW,KN,LC,VC,WS,SM,ST,SA,SN,RS,SC,SL,SG,SX,SK,SI,SB,SO,ZA,SS,SU,ES,LK,SD,SR,SE,CH,SY,TJ,TH,TL,TG,TO,TT,TN,TR,TM,TV,UG,UA,AE,GB,TZ,US,UY,UZ,VU,VE,VN,YE,YU,ZR,ZM,ZW,**"),
    "PA1b": UrlSpec(url_template="https://api.ipstatsdc.deda.prd.web1.wipo.int/api/v1/public/ips-search/downloadCsv?selectedTab=patent&indicator=12&reportType=15&fromYear={from_year}&toYear={to_year}&ipsOffSelValues=AF,OA,AP,AL,DZ,AD,AO,AG,AR,AM,AU,AT,AZ,BS,BH,BD,BB,BY,BE,BZ,BJ,BT,BO,BQ,BA,BW,BR,BN,BG,BF,BI,CV,KH,CM,CA,CF,TD,CL,CN,HK,MO,CO,KM,CG,CK,CR,CI,HR,CU,CW,CY,CZ,CS,KP,CD,DK,DJ,DM,DO,EC,EG,SV,GQ,ER,EE,SZ,ET,EA,EP,FJ,FI,FR,GA,GM,GE,DD,DE,GH,GR,GD,GT,GG,GN,GW,GY,HT,VA,HN,HU,IS,IN,ID,IR,IQ,IE,IL,IT,JM,JP,JO,KZ,KE,KI,KW,KG,LA,LV,LB,LS,LR,LY,LI,LT,LU,MG,MW,MY,MV,ML,MT,MH,MR,MU,MX,FM,MC,MN,ME,MA,MZ,MM,NA,NR,NP,NL,AN,NZ,NI,NE,NG,NU,MK,NO,OM,PK,PW,PA,PG,PY,GC,PE,PH,PL,PT,QA,KR,MD,RO,RU,RW,KN,LC,VC,WS,SM,ST,SA,SN,RS,SC,SL,SG,SX,SK,SI,SB,SO,ZA,SS,SU,ES,LK,SD,SR,SE,CH,SY,TJ,TH,TL,TG,TO,TT,TN,TR,TM,TV,UG,UA,AE,GB,TZ,US,UY,UZ,VU,VE,VN,YE,YU,ZR,ZM,ZW&ipsOriSelValues=AF,AL,DZ,AD,AO,AG,AR,AM,AU,AT,AZ,BS,BH,BD,BB,BY,BE,BZ,BJ,BT,BO,BQ,BA,BW,BR,BN,BG,BF,BI,CV,KH,CM,CA,CF,TD,CL,CN,HK,MO,CO,KM,CG,CK,CR,CI,HR,CU,CW,CY,CZ,CS,KP,CD,DK,DJ,DM,DO,EC,EG,SV,GQ,ER,EE,SZ,ET,FJ,FI,FR,GA,GM,GE,DD,DE,GH,GR,GD,GT,GN,GW,GY,HT,VA,HN,HU,IS,IN,ID,IR,IQ,IE,IL,IT,JM,JP,JO,KZ,KE,KI,KW,KG,LA,LV,LB,LS,LR,LY,LI,LT,LU,MG,MW,MY,MV,ML,MT,MH,MR,MU,MX,FM,MC,MN,ME,MA,MZ,MM,NA,NR,NP,NL,AN,NZ,NI,NE,NG,NU,MK,NO,OM,PK,PW,PA,PG,PY,PE,PH,PL,PT,QA,KR,MD,RO,RU,RW,KN,LC,VC,WS,SM,ST,SA,SN,RS,SC,SL,SG,SX,SK,SI,SB,SO,ZA,SS,SU,ES,LK,SD,SR,SE,CH,SY,TJ,TH,TL,TG,TO,TT,TN,TR,TM,TV,UG,UA,AE,GB,TZ,US,UY,UZ,VU,VE,VN,YE,YU,ZR,ZM,ZW,**"),
    "PA2a": UrlSpec(url_template="https://api.ipstatsdc.deda.prd.web1.wipo.int/api/v1/public/ips-search/downloadCsv?selectedTab=patent&indicator=13&reportType=15&fromYear={from_year}&toYear={to_year}&ipsOffSelValues=AF,OA,AP,AL,DZ,AD,AO,AG,AR,AM,AU,AT,AZ,BS,BH,BD,BB,BY,BE,BZ,BJ,BT,BO,BQ,BA,BW,BR,BN,BG,BF,BI,CV,KH,CM,CA,CF,TD,CL,CN,HK,MO,CO,KM,CG,CK,CR,CI,HR,CU,CW,CY,CZ,CS,KP,CD,DK,DJ,DM,DO,EC,EG,SV,GQ,ER,EE,SZ,ET,EA,EP,FJ,FI,FR,GA,GM,GE,DD,DE,GH,GR,GD,GT,GG,GN,GW,GY,HT,VA,HN,HU,IS,IN,ID,IR,IQ,IE,IL,IT,JM,JP,JO,KZ,KE,KI,KW,KG,LA,LV,LB,LS,LR,LY,LI,LT,LU,MG,MW,MY,MV,ML,MT,MH,MR,MU,MX,FM,MC,MN,ME,MA,MZ,MM,NA,NR,NP,NL,AN,NZ,NI,NE,NG,NU,MK,NO,OM,PK,PW,PA,PG,PY,GC,PE,PH,PL,PT,QA,KR,MD,RO,RU,RW,KN,LC,VC,WS,SM,ST,SA,SN,RS,SC,SL,SG,SX,SK,SI,SB,SO,ZA,SS,SU,ES,LK,SD,SR,SE,CH,SY,TJ,TH,TL,TG,TO,TT,TN,TR,TM,TV,UG,UA,AE,GB,TZ,US,UY,UZ,VU,VE,VN,YE,YU,ZR,ZM,ZW&ipsOriSelValues=AF,AL,DZ,AD,AO,AG,AR,AM,AU,AT,AZ,BS,BH,BD,BB,BY,BE,BZ,BJ,BT,BO,BQ,BA,BW,BR,BN,BG,BF,BI,CV,KH,CM,CA,CF,TD,CL,CN,HK,MO,CO,KM,CG,CK,CR,CI,HR,CU,CW,CY,CZ,CS,KP,CD,DK,DJ,DM,DO,EC,EG,SV,GQ,ER,EE,SZ,ET,FJ,FI,FR,GA,GM,GE,DD,DE,GH,GR,GD,GT,GN,GW,GY,HT,VA,HN,HU,IS,IN,ID,IR,IQ,IE,IL,IT,JM,JP,JO,KZ,KE,KI,KW,KG,LA,LV,LB,LS,LR,LY,LI,LT,LU,MG,MW,MY,MV,ML,MT,MH,MR,MU,MX,FM,MC,MN,ME,MA,MZ,MM,NA,NR,NP,NL,AN,NZ,NI,NE,NG,NU,MK,NO,OM,PK,PW,PA,PG,PY,PE,PH,PL,PT,QA,KR,MD,RO,RU,RW,KN,LC,VC,WS,SM,ST,SA,SN,RS,SC,SL,SG,SX,SK,SI,SB,SO,ZA,SS,SU,ES,LK,SD,SR,SE,CH,SY,TJ,TH,TL,TG,TO,TT,TN,TR,TM,TV,UG,UA,AE,GB,TZ,US,UY,UZ,VU,VE,VN,YE,YU,ZR,ZM,ZW,**"),
    "PA2b": UrlSpec(url_template="https://api.ipstatsdc.deda.prd.web1.wipo.int/api/v1/public/ips-search/downloadCsv?selectedTab=patent&indicator=14&reportType=15&fromYear={from_year}&toYear={to_year}&ipsOffSelValues=AF,OA,AP,AL,DZ,AD,AO,AG,AR,AM,AU,AT,AZ,BS,BH,BD,BB,BY,BE,BZ,BJ,BT,BO,BQ,BA,BW,BR,BN,BG,BF,BI,CV,KH,CM,CA,CF,TD,CL,CN,HK,MO,CO,KM,CG,CK,CR,CI,HR,CU,CW,CY,CZ,CS,KP,CD,DK,DJ,DM,DO,EC,EG,SV,GQ,ER,EE,SZ,ET,EA,EP,FJ,FI,FR,GA,GM,GE,DD,DE,GH,GR,GD,GT,GG,GN,GW,GY,HT,VA,HN,HU,IS,IN,ID,IR,IQ,IE,IL,IT,JM,JP,JO,KZ,KE,KI,KW,KG,LA,LV,LB,LS,LR,LY,LI,LT,LU,MG,MW,MY,MV,ML,MT,MH,MR,MU,MX,FM,MC,MN,ME,MA,MZ,MM,NA,NR,NP,NL,AN,NZ,NI,NE,NG,NU,MK,NO,OM,PK,PW,PA,PG,PY,GC,PE,PH,PL,PT,QA,KR,MD,RO,RU,RW,KN,LC,VC,WS,SM,ST,SA,SN,RS,SC,SL,SG,SX,SK,SI,SB,SO,ZA,SS,SU,ES,LK,SD,SR,SE,CH,SY,TJ,TH,TL,TG,TO,TT,TN,TR,TM,TV,UG,UA,AE,GB,TZ,US,UY,UZ,VU,VE,VN,YE,YU,ZR,ZM,ZW&ipsOriSelValues=AF,AL,DZ,AD,AO,AG,AR,AM,AU,AT,AZ,BS,BH,BD,BB,BY,BE,BZ,BJ,BT,BO,BQ,BA,BW,BR,BN,BG,BF,BI,CV,KH,CM,CA,CF,TD,CL,CN,HK,MO,CO,KM,CG,CK,CR,CI,HR,CU,CW,CY,CZ,CS,KP,CD,DK,DJ,DM,DO,EC,EG,SV,GQ,ER,EE,SZ,ET,FJ,FI,FR,GA,GM,GE,DD,DE,GH,GR,GD,GT,GN,GW,GY,HT,VA,HN,HU,IS,IN,ID,IR,IQ,IE,IL,IT,JM,JP,JO,KZ,KE,KI,KW,KG,LA,LV,LB,LS,LR,LY,LI,LT,LU,MG,MW,MY,MV,ML,MT,MH,MR,MU,MX,FM,MC,MN,ME,MA,MZ,MM,NA,NR,NP,NL,AN,NZ,NI,NE,NG,NU,MK,NO,OM,PK,PW,PA,PG,PY,PE,PH,PL,PT,QA,KR,MD,RO,RU,RW,KN,LC,VC,WS,SM,ST,SA,SN,RS,SC,SL,SG,SX,SK,SI,SB,SO,ZA,SS,SU,ES,LK,SD,SR,SE,CH,SY,TJ,TH,TL,TG,TO,TT,TN,TR,TM,TV,UG,UA,AE,GB,TZ,US,UY,UZ,VU,VE,VN,YE,YU,ZR,ZM,ZW,**"),
    "PA3":  UrlSpec(url_template="https://api.ipstatsdc.deda.prd.web1.wipo.int/api/v1/public/ips-search/downloadCsv?selectedTab=patent&indicator=15&reportType=11&fromYear={from_year}&toYear={to_year}&ipsOffSelValues=AF,OA,AP,AL,DZ,AD,AO,AG,AR,AM,AU,AT,AZ,BS,BH,BD,BB,BY,BE,BZ,BJ,BT,BO,BQ,BA,BW,BR,BN,BG,BF,BI,CV,KH,CM,CA,CF,TD,CL,CN,HK,MO,CO,KM,CG,CK,CR,CI,HR,CU,CW,CY,CZ,CS,KP,CD,DK,DJ,DM,DO,EC,EG,SV,GQ,ER,EE,SZ,ET,EA,EP,FJ,FI,FR,GA,GM,GE,DD,DE,GH,GR,GD,GT,GG,GN,GW,GY,HT,VA,HN,HU,IS,IN,ID,IR,IQ,IE,IL,IT,JM,JP,JO,KZ,KE,KI,KW,KG,LA,LV,LB,LS,LR,LY,LI,LT,LU,MG,MW,MY,MV,ML,MT,MH,MR,MU,MX,FM,MC,MN,ME,MA,MZ,MM,NA,NR,NP,NL,AN,NZ,NI,NE,NG,NU,MK,NO,OM,PK,PW,PA,PG,PY,GC,PE,PH,PL,PT,QA,KR,MD,RO,RU,RW,KN,LC,VC,WS,SM,ST,SA,SN,RS,SC,SL,SG,SX,SK,SI,SB,SO,ZA,SS,SU,ES,LK,SD,SR,SE,CH,SY,TJ,TH,TL,TG,TO,TT,TN,TR,TM,TV,UG,UA,AE,GB,TZ,US,UY,UZ,VU,VE,VN,YE,YU,ZR,ZM,ZW&ipsTechSelValues=900"),

    #fct_ip_flows_by_class_yearly
    "PA4a": UrlSpec(url_template="https://api.ipstatsdc.deda.prd.web1.wipo.int/api/v1/public/ips-search/downloadCsv?selectedTab=patent&indicator=17&reportType=15&fromYear={from_year}&toYear={to_year}&ipsOffSelValues=AF,OA,AP,AL,DZ,AD,AO,AG,AR,AM,AU,AT,AZ,BS,BH,BD,BB,BY,BE,BZ,BJ,BT,BO,BQ,BA,BW,BR,BN,BG,BF,BI,CV,KH,CM,CA,CF,TD,CL,CN,HK,MO,CO,KM,CG,CK,CR,CI,HR,CU,CW,CY,CZ,CS,KP,CD,DK,DJ,DM,DO,EC,EG,SV,GQ,ER,EE,SZ,ET,EA,EP,FJ,FI,FR,GA,GM,GE,DD,DE,GH,GR,GD,GT,GG,GN,GW,GY,HT,VA,HN,HU,IS,IN,ID,IR,IQ,IE,IL,IT,JM,JP,JO,KZ,KE,KI,KW,KG,LA,LV,LB,LS,LR,LY,LI,LT,LU,MG,MW,MY,MV,ML,MT,MH,MR,MU,MX,FM,MC,MN,ME,MA,MZ,MM,NA,NR,NP,NL,AN,NZ,NI,NE,NG,NU,MK,NO,OM,PK,PW,PA,PG,PY,GC,PE,PH,PL,PT,QA,KR,MD,RO,RU,RW,KN,LC,VC,WS,SM,ST,SA,SN,RS,SC,SL,SG,SX,SK,SI,SB,SO,ZA,SS,SU,ES,LK,SD,SR,SE,CH,SY,TJ,TH,TL,TG,TO,TT,TN,TR,TM,TV,UG,UA,AE,GB,TZ,US,UY,UZ,VU,VE,VN,YE,YU,ZR,ZM,ZW&ipsOriSelValues=AF,AL,DZ,AD,AO,AG,AR,AM,AU,AT,AZ,BS,BH,BD,BB,BY,BE,BZ,BJ,BT,BO,BQ,BA,BW,BR,BN,BG,BF,BI,CV,KH,CM,CA,CF,TD,CL,CN,HK,MO,CO,KM,CG,CK,CR,CI,HR,CU,CW,CY,CZ,CS,KP,CD,DK,DJ,DM,DO,EC,EG,SV,GQ,ER,EE,SZ,ET,FJ,FI,FR,GA,GM,GE,DD,DE,GH,GR,GD,GT,GN,GW,GY,HT,VA,HN,HU,IS,IN,ID,IR,IQ,IE,IL,IT,JM,JP,JO,KZ,KE,KI,KW,KG,LA,LV,LB,LS,LR,LY,LI,LT,LU,MG,MW,MY,MV,ML,MT,MH,MR,MU,MX,FM,MC,MN,ME,MA,MZ,MM,NA,NR,NP,NL,AN,NZ,NI,NE,NG,NU,MK,NO,OM,PK,PW,PA,PG,PY,PE,PH,PL,PT,QA,KR,MD,RO,RU,RW,KN,LC,VC,WS,SM,ST,SA,SN,RS,SC,SL,SG,SX,SK,SI,SB,SO,ZA,SS,SU,ES,LK,SD,SR,SE,CH,SY,TJ,TH,TL,TG,TO,TT,TN,TR,TM,TV,UG,UA,AE,GB,TZ,US,UY,UZ,VU,VE,VN,YE,YU,ZR,ZM,ZW,**&ipsTechSelValues=0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35"),
    "PA5":  UrlSpec(url_template="https://api.ipstatsdc.deda.prd.web1.wipo.int/api/v1/public/ips-search/downloadCsv?selectedTab=patent&indicator=24&reportType=15&fromYear={from_year}&toYear={to_year}&ipsOffSelValues=AF,OA,AP,AL,DZ,AD,AO,AG,AR,AM,AU,AT,AZ,BS,BH,BD,BB,BY,BE,BZ,BJ,BT,BO,BQ,BA,BW,BR,BN,BG,BF,BI,CV,KH,CM,CA,CF,TD,CL,CN,HK,MO,CO,KM,CG,CK,CR,CI,HR,CU,CW,CY,CZ,CS,KP,CD,DK,DJ,DM,DO,EC,EG,SV,GQ,ER,EE,SZ,ET,EA,EP,FJ,FI,FR,GA,GM,GE,DD,DE,GH,GR,GD,GT,GG,GN,GW,GY,HT,VA,HN,HU,IS,IN,ID,IR,IQ,IE,IL,IT,JM,JP,JO,KZ,KE,KI,KW,KG,LA,LV,LB,LS,LR,LY,LI,LT,LU,MG,MW,MY,MV,ML,MT,MH,MR,MU,MX,FM,MC,MN,ME,MA,MZ,MM,NA,NR,NP,NL,AN,NZ,NI,NE,NG,NU,MK,NO,OM,PK,PW,PA,PG,PY,GC,PE,PH,PL,PT,QA,KR,MD,RO,RU,RW,KN,LC,VC,WS,SM,ST,SA,SN,RS,SC,SL,SG,SX,SK,SI,SB,SO,ZA,SS,SU,ES,LK,SD,SR,SE,CH,SY,TJ,TH,TL,TG,TO,TT,TN,TR,TM,TV,UG,UA,AE,GB,TZ,US,UY,UZ,VU,VE,VN,YE,YU,ZR,ZM,ZW&ipsOriSelValues=AF,AL,DZ,AD,AO,AG,AR,AM,AU,AT,AZ,BS,BH,BD,BB,BY,BE,BZ,BJ,BT,BO,BQ,BA,BW,BR,BN,BG,BF,BI,CV,KH,CM,CA,CF,TD,CL,CN,HK,MO,CO,KM,CG,CK,CR,CI,HR,CU,CW,CY,CZ,CS,KP,CD,DK,DJ,DM,DO,EC,EG,SV,GQ,ER,EE,SZ,ET,FJ,FI,FR,GA,GM,GE,DD,DE,GH,GR,GD,GT,GN,GW,GY,HT,VA,HN,HU,IS,IN,ID,IR,IQ,IE,IL,IT,JM,JP,JO,KZ,KE,KI,KW,KG,LA,LV,LB,LS,LR,LY,LI,LT,LU,MG,MW,MY,MV,ML,MT,MH,MR,MU,MX,FM,MC,MN,ME,MA,MZ,MM,NA,NR,NP,NL,AN,NZ,NI,NE,NG,NU,MK,NO,OM,PK,PW,PA,PG,PY,PE,PH,PL,PT,QA,KR,MD,RO,RU,RW,KN,LC,VC,WS,SM,ST,SA,SN,RS,SC,SL,SG,SX,SK,SI,SB,SO,ZA,SS,SU,ES,LK,SD,SR,SE,CH,SY,TJ,TH,TL,TG,TO,TT,TN,TR,TM,TV,UG,UA,AE,GB,TZ,US,UY,UZ,VU,VE,VN,YE,YU,ZR,ZM,ZW,**&ipsTechSelValues=0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35"),

    #fct_ip_flows_by_residency
    "PA1a_origin": UrlSpec(url_template="https://api.ipstatsdc.deda.prd.web1.wipo.int/api/v1/public/ips-search/downloadCsv?selectedTab=patent&indicator=11&reportType=13&fromYear={from_year}&toYear={to_year}&ipsOriSelValues=AF,AL,DZ,AD,AO,AG,AR,AM,AU,AT,AZ,BS,BH,BD,BB,BY,BE,BZ,BJ,BT,BO,BQ,BA,BW,BR,BN,BG,BF,BI,CV,KH,CM,CA,CF,TD,CL,CN,HK,MO,CO,KM,CG,CK,CR,CI,HR,CU,CW,CY,CZ,CS,KP,CD,DK,DJ,DM,DO,EC,EG,SV,GQ,ER,EE,SZ,ET,FJ,FI,FR,GA,GM,GE,DD,DE,GH,GR,GD,GT,GN,GW,GY,HT,VA,HN,HU,IS,IN,ID,IR,IQ,IE,IL,IT,JM,JP,JO,KZ,KE,KI,KW,KG,LA,LV,LB,LS,LR,LY,LI,LT,LU,MG,MW,MY,MV,ML,MT,MH,MR,MU,MX,FM,MC,MN,ME,MA,MZ,MM,NA,NR,NP,NL,AN,NZ,NI,NE,NG,NU,MK,NO,OM,PK,PW,PA,PG,PY,PE,PH,PL,PT,QA,KR,MD,RO,RU,RW,KN,LC,VC,WS,SM,ST,SA,SN,RS,SC,SL,SG,SX,SK,SI,SB,SO,ZA,SS,SU,ES,LK,SD,SR,SE,CH,SY,TJ,TH,TL,TG,TO,TT,TN,TR,TM,TV,UG,UA,AE,GB,TZ,US,UY,UZ,VU,VE,VN,YE,YU,ZR,ZM,ZW&ipsTechSelValues=910,911,912,913,914"),
    "PA1b_origin": UrlSpec(url_template="https://api.ipstatsdc.deda.prd.web1.wipo.int/api/v1/public/ips-search/downloadCsv?selectedTab=patent&indicator=12&reportType=13&fromYear={from_year}&toYear={to_year}&ipsOriSelValues=AF,AL,DZ,AD,AO,AG,AR,AM,AU,AT,AZ,BS,BH,BD,BB,BY,BE,BZ,BJ,BT,BO,BQ,BA,BW,BR,BN,BG,BF,BI,CV,KH,CM,CA,CF,TD,CL,CN,HK,MO,CO,KM,CG,CK,CR,CI,HR,CU,CW,CY,CZ,CS,KP,CD,DK,DJ,DM,DO,EC,EG,SV,GQ,ER,EE,SZ,ET,FJ,FI,FR,GA,GM,GE,DD,DE,GH,GR,GD,GT,GN,GW,GY,HT,VA,HN,HU,IS,IN,ID,IR,IQ,IE,IL,IT,JM,JP,JO,KZ,KE,KI,KW,KG,LA,LV,LB,LS,LR,LY,LI,LT,LU,MG,MW,MY,MV,ML,MT,MH,MR,MU,MX,FM,MC,MN,ME,MA,MZ,MM,NA,NR,NP,NL,AN,NZ,NI,NE,NG,NU,MK,NO,OM,PK,PW,PA,PG,PY,PE,PH,PL,PT,QA,KR,MD,RO,RU,RW,KN,LC,VC,WS,SM,ST,SA,SN,RS,SC,SL,SG,SX,SK,SI,SB,SO,ZA,SS,SU,ES,LK,SD,SR,SE,CH,SY,TJ,TH,TL,TG,TO,TT,TN,TR,TM,TV,UG,UA,AE,GB,TZ,US,UY,UZ,VU,VE,VN,YE,YU,ZR,ZM,ZW&ipsTechSelValues=910,911,912,913,914"),
    "PA1a_office": UrlSpec(url_template="https://api.ipstatsdc.deda.prd.web1.wipo.int/api/v1/public/ips-search/downloadCsv?selectedTab=patent&indicator=11&reportType=11&fromYear={from_year}&toYear={to_year}&ipsOffSelValues=AF,OA,AP,AL,DZ,AD,AO,AG,AR,AM,AU,AT,AZ,BS,BH,BD,BB,BY,BE,BZ,BJ,BT,BO,BQ,BA,BW,BR,BN,BG,BF,BI,CV,KH,CM,CA,CF,TD,CL,CN,HK,MO,CO,KM,CG,CK,CR,CI,HR,CU,CW,CY,CZ,CS,KP,CD,DK,DJ,DM,DO,EC,EG,SV,GQ,ER,EE,SZ,ET,EA,EP,FJ,FI,FR,GA,GM,GE,DD,DE,GH,GR,GD,GT,GG,GN,GW,GY,HT,VA,HN,HU,IS,IN,ID,IR,IQ,IE,IL,IT,JM,JP,JO,KZ,KE,KI,KW,KG,LA,LV,LB,LS,LR,LY,LI,LT,LU,MG,MW,MY,MV,ML,MT,MH,MR,MU,MX,FM,MC,MN,ME,MA,MZ,MM,NA,NR,NP,NL,AN,NZ,NI,NE,NG,NU,MK,NO,OM,PK,PW,PA,PG,PY,GC,PE,PH,PL,PT,QA,KR,MD,RO,RU,RW,KN,LC,VC,WS,SM,ST,SA,SN,RS,SC,SL,SG,SX,SK,SI,SB,SO,ZA,SS,SU,ES,LK,SD,SR,SE,CH,SY,TJ,TH,TL,TG,TO,TT,TN,TR,TM,TV,UG,UA,AE,GB,TZ,US,UY,UZ,VU,VE,VN,YE,YU,ZR,ZM,ZW,WD,R1,R2,R3,R4,R5,R6&ipsTechSelValues=900,901,902"),
    "PA1b_office": UrlSpec(url_template="https://api.ipstatsdc.deda.prd.web1.wipo.int/api/v1/public/ips-search/downloadCsv?selectedTab=patent&indicator=12&reportType=11&fromYear={from_year}&toYear={to_year}&ipsOffSelValues=AF,OA,AP,AL,DZ,AD,AO,AG,AR,AM,AU,AT,AZ,BS,BH,BD,BB,BY,BE,BZ,BJ,BT,BO,BQ,BA,BW,BR,BN,BG,BF,BI,CV,KH,CM,CA,CF,TD,CL,CN,HK,MO,CO,KM,CG,CK,CR,CI,HR,CU,CW,CY,CZ,CS,KP,CD,DK,DJ,DM,DO,EC,EG,SV,GQ,ER,EE,SZ,ET,EA,EP,FJ,FI,FR,GA,GM,GE,DD,DE,GH,GR,GD,GT,GG,GN,GW,GY,HT,VA,HN,HU,IS,IN,ID,IR,IQ,IE,IL,IT,JM,JP,JO,KZ,KE,KI,KW,KG,LA,LV,LB,LS,LR,LY,LI,LT,LU,MG,MW,MY,MV,ML,MT,MH,MR,MU,MX,FM,MC,MN,ME,MA,MZ,MM,NA,NR,NP,NL,AN,NZ,NI,NE,NG,NU,MK,NO,OM,PK,PW,PA,PG,PY,GC,PE,PH,PL,PT,QA,KR,MD,RO,RU,RW,KN,LC,VC,WS,SM,ST,SA,SN,RS,SC,SL,SG,SX,SK,SI,SB,SO,ZA,SS,SU,ES,LK,SD,SR,SE,CH,SY,TJ,TH,TL,TG,TO,TT,TN,TR,TM,TV,UG,UA,AE,GB,TZ,US,UY,UZ,VU,VE,VN,YE,YU,ZR,ZM,ZW,WD,R1,R2,R3,R4,R5,R6&ipsTechSelValues=900,901,902"),

    "PA2a_origin": UrlSpec(url_template="https://api.ipstatsdc.deda.prd.web1.wipo.int/api/v1/public/ips-search/downloadCsv?selectedTab=patent&indicator=13&reportType=13&fromYear={from_year}&toYear={to_year}&ipsOriSelValues=AF,AL,DZ,AD,AO,AG,AR,AM,AU,AT,AZ,BS,BH,BD,BB,BY,BE,BZ,BJ,BT,BO,BQ,BA,BW,BR,BN,BG,BF,BI,CV,KH,CM,CA,CF,TD,CL,CN,HK,MO,CO,KM,CG,CK,CR,CI,HR,CU,CW,CY,CZ,CS,KP,CD,DK,DJ,DM,DO,EC,EG,SV,GQ,ER,EE,SZ,ET,FJ,FI,FR,GA,GM,GE,DD,DE,GH,GR,GD,GT,GN,GW,GY,HT,VA,HN,HU,IS,IN,ID,IR,IQ,IE,IL,IT,JM,JP,JO,KZ,KE,KI,KW,KG,LA,LV,LB,LS,LR,LY,LI,LT,LU,MG,MW,MY,MV,ML,MT,MH,MR,MU,MX,FM,MC,MN,ME,MA,MZ,MM,NA,NR,NP,NL,AN,NZ,NI,NE,NG,NU,MK,NO,OM,PK,PW,PA,PG,PY,PE,PH,PL,PT,QA,KR,MD,RO,RU,RW,KN,LC,VC,WS,SM,ST,SA,SN,RS,SC,SL,SG,SX,SK,SI,SB,SO,ZA,SS,SU,ES,LK,SD,SR,SE,CH,SY,TJ,TH,TL,TG,TO,TT,TN,TR,TM,TV,UG,UA,AE,GB,TZ,US,UY,UZ,VU,VE,VN,YE,YU,ZR,ZM,ZW&ipsTechSelValues=910,911,912,913,914"),
    "PA2b_origin": UrlSpec(url_template="https://api.ipstatsdc.deda.prd.web1.wipo.int/api/v1/public/ips-search/downloadCsv?selectedTab=patent&indicator=14&reportType=13&fromYear={from_year}&toYear={to_year}&ipsOriSelValues=AF,AL,DZ,AD,AO,AG,AR,AM,AU,AT,AZ,BS,BH,BD,BB,BY,BE,BZ,BJ,BT,BO,BQ,BA,BW,BR,BN,BG,BF,BI,CV,KH,CM,CA,CF,TD,CL,CN,HK,MO,CO,KM,CG,CK,CR,CI,HR,CU,CW,CY,CZ,CS,KP,CD,DK,DJ,DM,DO,EC,EG,SV,GQ,ER,EE,SZ,ET,FJ,FI,FR,GA,GM,GE,DD,DE,GH,GR,GD,GT,GN,GW,GY,HT,VA,HN,HU,IS,IN,ID,IR,IQ,IE,IL,IT,JM,JP,JO,KZ,KE,KI,KW,KG,LA,LV,LB,LS,LR,LY,LI,LT,LU,MG,MW,MY,MV,ML,MT,MH,MR,MU,MX,FM,MC,MN,ME,MA,MZ,MM,NA,NR,NP,NL,AN,NZ,NI,NE,NG,NU,MK,NO,OM,PK,PW,PA,PG,PY,PE,PH,PL,PT,QA,KR,MD,RO,RU,RW,KN,LC,VC,WS,SM,ST,SA,SN,RS,SC,SL,SG,SX,SK,SI,SB,SO,ZA,SS,SU,ES,LK,SD,SR,SE,CH,SY,TJ,TH,TL,TG,TO,TT,TN,TR,TM,TV,UG,UA,AE,GB,TZ,US,UY,UZ,VU,VE,VN,YE,YU,ZR,ZM,ZW&ipsTechSelValues=910,911,912,913,914"),
    "PA2a_office": UrlSpec(url_template="https://api.ipstatsdc.deda.prd.web1.wipo.int/api/v1/public/ips-search/downloadCsv?selectedTab=patent&indicator=13&reportType=11&fromYear={from_year}&toYear={to_year}&ipsOffSelValues=AF,OA,AP,AL,DZ,AD,AO,AG,AR,AM,AU,AT,AZ,BS,BH,BD,BB,BY,BE,BZ,BJ,BT,BO,BQ,BA,BW,BR,BN,BG,BF,BI,CV,KH,CM,CA,CF,TD,CL,CN,HK,MO,CO,KM,CG,CK,CR,CI,HR,CU,CW,CY,CZ,CS,KP,CD,DK,DJ,DM,DO,EC,EG,SV,GQ,ER,EE,SZ,ET,EA,EP,FJ,FI,FR,GA,GM,GE,DD,DE,GH,GR,GD,GT,GG,GN,GW,GY,HT,VA,HN,HU,IS,IN,ID,IR,IQ,IE,IL,IT,JM,JP,JO,KZ,KE,KI,KW,KG,LA,LV,LB,LS,LR,LY,LI,LT,LU,MG,MW,MY,MV,ML,MT,MH,MR,MU,MX,FM,MC,MN,ME,MA,MZ,MM,NA,NR,NP,NL,AN,NZ,NI,NE,NG,NU,MK,NO,OM,PK,PW,PA,PG,PY,GC,PE,PH,PL,PT,QA,KR,MD,RO,RU,RW,KN,LC,VC,WS,SM,ST,SA,SN,RS,SC,SL,SG,SX,SK,SI,SB,SO,ZA,SS,SU,ES,LK,SD,SR,SE,CH,SY,TJ,TH,TL,TG,TO,TT,TN,TR,TM,TV,UG,UA,AE,GB,TZ,US,UY,UZ,VU,VE,VN,YE,YU,ZR,ZM,ZW,WD,R1,R2,R3,R4,R5,R6&ipsTechSelValues=900,901,902"),
    "PA2b_office": UrlSpec(url_template="https://api.ipstatsdc.deda.prd.web1.wipo.int/api/v1/public/ips-search/downloadCsv?selectedTab=patent&indicator=14&reportType=11&fromYear={from_year}&toYear={to_year}&ipsOffSelValues=AF,OA,AP,AL,DZ,AD,AO,AG,AR,AM,AU,AT,AZ,BS,BH,BD,BB,BY,BE,BZ,BJ,BT,BO,BQ,BA,BW,BR,BN,BG,BF,BI,CV,KH,CM,CA,CF,TD,CL,CN,HK,MO,CO,KM,CG,CK,CR,CI,HR,CU,CW,CY,CZ,CS,KP,CD,DK,DJ,DM,DO,EC,EG,SV,GQ,ER,EE,SZ,ET,EA,EP,FJ,FI,FR,GA,GM,GE,DD,DE,GH,GR,GD,GT,GG,GN,GW,GY,HT,VA,HN,HU,IS,IN,ID,IR,IQ,IE,IL,IT,JM,JP,JO,KZ,KE,KI,KW,KG,LA,LV,LB,LS,LR,LY,LI,LT,LU,MG,MW,MY,MV,ML,MT,MH,MR,MU,MX,FM,MC,MN,ME,MA,MZ,MM,NA,NR,NP,NL,AN,NZ,NI,NE,NG,NU,MK,NO,OM,PK,PW,PA,PG,PY,GC,PE,PH,PL,PT,QA,KR,MD,RO,RU,RW,KN,LC,VC,WS,SM,ST,SA,SN,RS,SC,SL,SG,SX,SK,SI,SB,SO,ZA,SS,SU,ES,LK,SD,SR,SE,CH,SY,TJ,TH,TL,TG,TO,TT,TN,TR,TM,TV,UG,UA,AE,GB,TZ,US,UY,UZ,VU,VE,VN,YE,YU,ZR,ZM,ZW,WD,R1,R2,R3,R4,R5,R6&ipsTechSelValues=900,901,902"),


##################### TRADEMARKS ###########################################################################################################
    # fct_ip_flows_yearly
    "TM1a": UrlSpec(url_template="https://api.ipstatsdc.deda.prd.web1.wipo.int/api/v1/public/ips-search/downloadCsv?selectedTab=trademark&indicator=31&reportType=15&fromYear={from_year}&toYear={to_year}&ipsOffSelValues=AF,OA,AP,AL,DZ,AD,AO,AG,AR,AM,AU,AT,AZ,BS,BH,BD,BB,BY,BE,BZ,BX,BJ,BT,BO,BQ,BA,BW,BR,BN,BG,BF,BI,CV,KH,CM,CA,CF,TD,CL,CN,HK,MO,CO,KM,CG,CK,CR,CI,HR,CU,CW,CY,CZ,CS,KP,CD,DK,DJ,DM,DO,EC,EG,SV,GQ,ER,EE,SZ,ET,EM,FJ,FI,FR,GA,GM,GE,DD,DE,GH,GR,GD,GT,GG,GN,GW,GY,HT,VA,HN,HU,IS,IN,ID,IR,IQ,IE,IL,IT,JM,JP,JO,KZ,KE,KI,KW,KG,LA,LV,LB,LS,LR,LY,LI,LT,LU,MG,MW,MY,MV,ML,MT,MH,MR,MU,MX,FM,MC,MN,ME,MA,MZ,MM,NA,NR,NP,NL,AN,NZ,NI,NE,NG,NU,MK,NO,OM,PK,PW,PA,PG,PY,PE,PH,PL,PT,QA,KR,MD,RO,RU,RW,KN,LC,VC,WS,SM,ST,SA,SN,RS,SC,SL,SG,SX,SK,SI,SB,SO,ZA,SS,SU,ES,LK,SD,SR,SE,CH,SY,TJ,TH,TL,TG,TO,TT,TN,TR,TM,TV,UG,UA,AE,GB,TZ,US,UY,UZ,VU,VE,VN,YE,YU,ZR,ZM,ZW&ipsOriSelValues=AF,AL,DZ,AD,AO,AG,AR,AM,AU,AT,AZ,BS,BH,BD,BB,BY,BE,BZ,BJ,BT,BO,BQ,BA,BW,BR,BN,BG,BF,BI,CV,KH,CM,CA,CF,TD,CL,CN,HK,MO,CO,KM,CG,CK,CR,CI,HR,CU,CW,CY,CZ,CS,KP,CD,DK,DJ,DM,DO,EC,EG,SV,GQ,ER,EE,SZ,ET,FJ,FI,FR,GA,GM,GE,DD,DE,GH,GR,GD,GT,GN,GW,GY,HT,VA,HN,HU,IS,IN,ID,IR,IQ,IE,IL,IT,JM,JP,JO,KZ,KE,KI,KW,KG,LA,LV,LB,LS,LR,LY,LI,LT,LU,MG,MW,MY,MV,ML,MT,MH,MR,MU,MX,FM,MC,MN,ME,MA,MZ,MM,NA,NR,NP,NL,AN,NZ,NI,NE,NG,NU,MK,NO,OM,PK,PW,PA,PG,PY,PE,PH,PL,PT,QA,KR,MD,RO,RU,RW,KN,LC,VC,WS,SM,ST,SA,SN,RS,SC,SL,SG,SX,SK,SI,SB,SO,ZA,SS,SU,ES,LK,SD,SR,SE,CH,SY,TJ,TH,TL,TG,TO,TT,TN,TR,TM,TV,UG,UA,AE,GB,TZ,US,UY,UZ,VU,VE,VN,YE,YU,ZR,ZM,ZW,**"),
    "TM1b": UrlSpec(url_template="https://api.ipstatsdc.deda.prd.web1.wipo.int/api/v1/public/ips-search/downloadCsv?selectedTab=trademark&indicator=32&reportType=15&fromYear={from_year}&toYear={to_year}&ipsOffSelValues=AF,OA,AP,AL,DZ,AD,AO,AG,AR,AM,AU,AT,AZ,BS,BH,BD,BB,BY,BE,BZ,BX,BJ,BT,BO,BQ,BA,BW,BR,BN,BG,BF,BI,CV,KH,CM,CA,CF,TD,CL,CN,HK,MO,CO,KM,CG,CK,CR,CI,HR,CU,CW,CY,CZ,CS,KP,CD,DK,DJ,DM,DO,EC,EG,SV,GQ,ER,EE,SZ,ET,EM,FJ,FI,FR,GA,GM,GE,DD,DE,GH,GR,GD,GT,GG,GN,GW,GY,HT,VA,HN,HU,IS,IN,ID,IR,IQ,IE,IL,IT,JM,JP,JO,KZ,KE,KI,KW,KG,LA,LV,LB,LS,LR,LY,LI,LT,LU,MG,MW,MY,MV,ML,MT,MH,MR,MU,MX,FM,MC,MN,ME,MA,MZ,MM,NA,NR,NP,NL,AN,NZ,NI,NE,NG,NU,MK,NO,OM,PK,PW,PA,PG,PY,PE,PH,PL,PT,QA,KR,MD,RO,RU,RW,KN,LC,VC,WS,SM,ST,SA,SN,RS,SC,SL,SG,SX,SK,SI,SB,SO,ZA,SS,SU,ES,LK,SD,SR,SE,CH,SY,TJ,TH,TL,TG,TO,TT,TN,TR,TM,TV,UG,UA,AE,GB,TZ,US,UY,UZ,VU,VE,VN,YE,YU,ZR,ZM,ZW&ipsOriSelValues=AF,AL,DZ,AD,AO,AG,AR,AM,AU,AT,AZ,BS,BH,BD,BB,BY,BE,BZ,BJ,BT,BO,BQ,BA,BW,BR,BN,BG,BF,BI,CV,KH,CM,CA,CF,TD,CL,CN,HK,MO,CO,KM,CG,CK,CR,CI,HR,CU,CW,CY,CZ,CS,KP,CD,DK,DJ,DM,DO,EC,EG,SV,GQ,ER,EE,SZ,ET,FJ,FI,FR,GA,GM,GE,DD,DE,GH,GR,GD,GT,GN,GW,GY,HT,VA,HN,HU,IS,IN,ID,IR,IQ,IE,IL,IT,JM,JP,JO,KZ,KE,KI,KW,KG,LA,LV,LB,LS,LR,LY,LI,LT,LU,MG,MW,MY,MV,ML,MT,MH,MR,MU,MX,FM,MC,MN,ME,MA,MZ,MM,NA,NR,NP,NL,AN,NZ,NI,NE,NG,NU,MK,NO,OM,PK,PW,PA,PG,PY,PE,PH,PL,PT,QA,KR,MD,RO,RU,RW,KN,LC,VC,WS,SM,ST,SA,SN,RS,SC,SL,SG,SX,SK,SI,SB,SO,ZA,SS,SU,ES,LK,SD,SR,SE,CH,SY,TJ,TH,TL,TG,TO,TT,TN,TR,TM,TV,UG,UA,AE,GB,TZ,US,UY,UZ,VU,VE,VN,YE,YU,ZR,ZM,ZW,**"),
    "TM2a": UrlSpec(url_template="https://api.ipstatsdc.deda.prd.web1.wipo.int/api/v1/public/ips-search/downloadCsv?selectedTab=trademark&indicator=33&reportType=15&fromYear={from_year}&toYear={to_year}&ipsOffSelValues=AF,OA,AP,AL,DZ,AD,AO,AG,AR,AM,AU,AT,AZ,BS,BH,BD,BB,BY,BE,BZ,BX,BJ,BT,BO,BQ,BA,BW,BR,BN,BG,BF,BI,CV,KH,CM,CA,CF,TD,CL,CN,HK,MO,CO,KM,CG,CK,CR,CI,HR,CU,CW,CY,CZ,CS,KP,CD,DK,DJ,DM,DO,EC,EG,SV,GQ,ER,EE,SZ,ET,EM,FJ,FI,FR,GA,GM,GE,DD,DE,GH,GR,GD,GT,GG,GN,GW,GY,HT,VA,HN,HU,IS,IN,ID,IR,IQ,IE,IL,IT,JM,JP,JO,KZ,KE,KI,KW,KG,LA,LV,LB,LS,LR,LY,LI,LT,LU,MG,MW,MY,MV,ML,MT,MH,MR,MU,MX,FM,MC,MN,ME,MA,MZ,MM,NA,NR,NP,NL,AN,NZ,NI,NE,NG,NU,MK,NO,OM,PK,PW,PA,PG,PY,PE,PH,PL,PT,QA,KR,MD,RO,RU,RW,KN,LC,VC,WS,SM,ST,SA,SN,RS,SC,SL,SG,SX,SK,SI,SB,SO,ZA,SS,SU,ES,LK,SD,SR,SE,CH,SY,TJ,TH,TL,TG,TO,TT,TN,TR,TM,TV,UG,UA,AE,GB,TZ,US,UY,UZ,VU,VE,VN,YE,YU,ZR,ZM,ZW&ipsOriSelValues=AF,AL,DZ,AD,AO,AG,AR,AM,AU,AT,AZ,BS,BH,BD,BB,BY,BE,BZ,BJ,BT,BO,BQ,BA,BW,BR,BN,BG,BF,BI,CV,KH,CM,CA,CF,TD,CL,CN,HK,MO,CO,KM,CG,CK,CR,CI,HR,CU,CW,CY,CZ,CS,KP,CD,DK,DJ,DM,DO,EC,EG,SV,GQ,ER,EE,SZ,ET,FJ,FI,FR,GA,GM,GE,DD,DE,GH,GR,GD,GT,GN,GW,GY,HT,VA,HN,HU,IS,IN,ID,IR,IQ,IE,IL,IT,JM,JP,JO,KZ,KE,KI,KW,KG,LA,LV,LB,LS,LR,LY,LI,LT,LU,MG,MW,MY,MV,ML,MT,MH,MR,MU,MX,FM,MC,MN,ME,MA,MZ,MM,NA,NR,NP,NL,AN,NZ,NI,NE,NG,NU,MK,NO,OM,PK,PW,PA,PG,PY,PE,PH,PL,PT,QA,KR,MD,RO,RU,RW,KN,LC,VC,WS,SM,ST,SA,SN,RS,SC,SL,SG,SX,SK,SI,SB,SO,ZA,SS,SU,ES,LK,SD,SR,SE,CH,SY,TJ,TH,TL,TG,TO,TT,TN,TR,TM,TV,UG,UA,AE,GB,TZ,US,UY,UZ,VU,VE,VN,YE,YU,ZR,ZM,ZW,**"),
    "TM2b": UrlSpec(url_template="https://api.ipstatsdc.deda.prd.web1.wipo.int/api/v1/public/ips-search/downloadCsv?selectedTab=trademark&indicator=34&reportType=15&fromYear={from_year}&toYear={to_year}&ipsOffSelValues=AF,OA,AP,AL,DZ,AD,AO,AG,AR,AM,AU,AT,AZ,BS,BH,BD,BB,BY,BE,BZ,BX,BJ,BT,BO,BQ,BA,BW,BR,BN,BG,BF,BI,CV,KH,CM,CA,CF,TD,CL,CN,HK,MO,CO,KM,CG,CK,CR,CI,HR,CU,CW,CY,CZ,CS,KP,CD,DK,DJ,DM,DO,EC,EG,SV,GQ,ER,EE,SZ,ET,EM,FJ,FI,FR,GA,GM,GE,DD,DE,GH,GR,GD,GT,GG,GN,GW,GY,HT,VA,HN,HU,IS,IN,ID,IR,IQ,IE,IL,IT,JM,JP,JO,KZ,KE,KI,KW,KG,LA,LV,LB,LS,LR,LY,LI,LT,LU,MG,MW,MY,MV,ML,MT,MH,MR,MU,MX,FM,MC,MN,ME,MA,MZ,MM,NA,NR,NP,NL,AN,NZ,NI,NE,NG,NU,MK,NO,OM,PK,PW,PA,PG,PY,PE,PH,PL,PT,QA,KR,MD,RO,RU,RW,KN,LC,VC,WS,SM,ST,SA,SN,RS,SC,SL,SG,SX,SK,SI,SB,SO,ZA,SS,SU,ES,LK,SD,SR,SE,CH,SY,TJ,TH,TL,TG,TO,TT,TN,TR,TM,TV,UG,UA,AE,GB,TZ,US,UY,UZ,VU,VE,VN,YE,YU,ZR,ZM,ZW&ipsOriSelValues=AF,AL,DZ,AD,AO,AG,AR,AM,AU,AT,AZ,BS,BH,BD,BB,BY,BE,BZ,BJ,BT,BO,BQ,BA,BW,BR,BN,BG,BF,BI,CV,KH,CM,CA,CF,TD,CL,CN,HK,MO,CO,KM,CG,CK,CR,CI,HR,CU,CW,CY,CZ,CS,KP,CD,DK,DJ,DM,DO,EC,EG,SV,GQ,ER,EE,SZ,ET,FJ,FI,FR,GA,GM,GE,DD,DE,GH,GR,GD,GT,GN,GW,GY,HT,VA,HN,HU,IS,IN,ID,IR,IQ,IE,IL,IT,JM,JP,JO,KZ,KE,KI,KW,KG,LA,LV,LB,LS,LR,LY,LI,LT,LU,MG,MW,MY,MV,ML,MT,MH,MR,MU,MX,FM,MC,MN,ME,MA,MZ,MM,NA,NR,NP,NL,AN,NZ,NI,NE,NG,NU,MK,NO,OM,PK,PW,PA,PG,PY,PE,PH,PL,PT,QA,KR,MD,RO,RU,RW,KN,LC,VC,WS,SM,ST,SA,SN,RS,SC,SL,SG,SX,SK,SI,SB,SO,ZA,SS,SU,ES,LK,SD,SR,SE,CH,SY,TJ,TH,TL,TG,TO,TT,TN,TR,TM,TV,UG,UA,AE,GB,TZ,US,UY,UZ,VU,VE,VN,YE,YU,ZR,ZM,ZW,**"),
    "TM7":  UrlSpec(url_template="https://api.ipstatsdc.deda.prd.web1.wipo.int/api/v1/public/ips-search/downloadCsv?selectedTab=trademark&indicator=43&reportType=11&fromYear={from_year}&toYear={to_year}&ipsOffSelValues=AF,OA,AP,AL,DZ,AD,AO,AG,AR,AM,AU,AT,AZ,BS,BH,BD,BB,BY,BE,BZ,BX,BJ,BT,BO,BQ,BA,BW,BR,BN,BG,BF,BI,CV,KH,CM,CA,CF,TD,CL,CN,HK,MO,CO,KM,CG,CK,CR,CI,HR,CU,CW,CY,CZ,CS,KP,CD,DK,DJ,DM,DO,EC,EG,SV,GQ,ER,EE,SZ,ET,EM,FJ,FI,FR,GA,GM,GE,DD,DE,GH,GR,GD,GT,GG,GN,GW,GY,HT,VA,HN,HU,IS,IN,ID,IR,IQ,IE,IL,IT,JM,JP,JO,KZ,KE,KI,KW,KG,LA,LV,LB,LS,LR,LY,LI,LT,LU,MG,MW,MY,MV,ML,MT,MH,MR,MU,MX,FM,MC,MN,ME,MA,MZ,MM,NA,NR,NP,NL,AN,NZ,NI,NE,NG,NU,MK,NO,OM,PK,PW,PA,PG,PY,PE,PH,PL,PT,QA,KR,MD,RO,RU,RW,KN,LC,VC,WS,SM,ST,SA,SN,RS,SC,SL,SG,SX,SK,SI,SB,SO,ZA,SS,SU,ES,LK,SD,SR,SE,CH,SY,TJ,TH,TL,TG,TO,TT,TN,TR,TM,TV,UG,UA,AE,GB,TZ,US,UY,UZ,VU,VE,VN,YE,YU,ZR,ZM,ZW"),

    #fct_ip_flows_by_class_yearly
    "TM3a": UrlSpec(url_template="https://api.ipstatsdc.deda.prd.web1.wipo.int/api/v1/public/ips-search/downloadCsv?selectedTab=trademark&indicator=35&reportType=15&fromYear={from_year}&toYear={to_year}&ipsOffSelValues=AF,OA,AP,AL,DZ,AD,AO,AG,AR,AM,AU,AT,AZ,BS,BH,BD,BB,BY,BE,BZ,BX,BJ,BT,BO,BQ,BA,BW,BR,BN,BG,BF,BI,CV,KH,CM,CA,CF,TD,CL,CN,HK,MO,CO,KM,CG,CK,CR,CI,HR,CU,CW,CY,CZ,CS,KP,CD,DK,DJ,DM,DO,EC,EG,SV,GQ,ER,EE,SZ,ET,EM,FJ,FI,FR,GA,GM,GE,DD,DE,GH,GR,GD,GT,GG,GN,GW,GY,HT,VA,HN,HU,IS,IN,ID,IR,IQ,IE,IL,IT,JM,JP,JO,KZ,KE,KI,KW,KG,LA,LV,LB,LS,LR,LY,LI,LT,LU,MG,MW,MY,MV,ML,MT,MH,MR,MU,MX,FM,MC,MN,ME,MA,MZ,MM,NA,NR,NP,NL,AN,NZ,NI,NE,NG,NU,MK,NO,OM,PK,PW,PA,PG,PY,PE,PH,PL,PT,QA,KR,MD,RO,RU,RW,KN,LC,VC,WS,SM,ST,SA,SN,RS,SC,SL,SG,SX,SK,SI,SB,SO,ZA,SS,SU,ES,LK,SD,SR,SE,CH,SY,TJ,TH,TL,TG,TO,TT,TN,TR,TM,TV,UG,UA,AE,GB,TZ,US,UY,UZ,VU,VE,VN,YE,YU,ZR,ZM,ZW&ipsOriSelValues=AF,AL,DZ,AD,AO,AG,AR,AM,AU,AT,AZ,BS,BH,BD,BB,BY,BE,BZ,BJ,BT,BO,BQ,BA,BW,BR,BN,BG,BF,BI,CV,KH,CM,CA,CF,TD,CL,CN,HK,MO,CO,KM,CG,CK,CR,CI,HR,CU,CW,CY,CZ,CS,KP,CD,DK,DJ,DM,DO,EC,EG,SV,GQ,ER,EE,SZ,ET,FJ,FI,FR,GA,GM,GE,DD,DE,GH,GR,GD,GT,GN,GW,GY,HT,VA,HN,HU,IS,IN,ID,IR,IQ,IE,IL,IT,JM,JP,JO,KZ,KE,KI,KW,KG,LA,LV,LB,LS,LR,LY,LI,LT,LU,MG,MW,MY,MV,ML,MT,MH,MR,MU,MX,FM,MC,MN,ME,MA,MZ,MM,NA,NR,NP,NL,AN,NZ,NI,NE,NG,NU,MK,NO,OM,PK,PW,PA,PG,PY,PE,PH,PL,PT,QA,KR,MD,RO,RU,RW,KN,LC,VC,WS,SM,ST,SA,SN,RS,SC,SL,SG,SX,SK,SI,SB,SO,ZA,SS,SU,ES,LK,SD,SR,SE,CH,SY,TJ,TH,TL,TG,TO,TT,TN,TR,TM,TV,UG,UA,AE,GB,TZ,US,UY,UZ,VU,VE,VN,YE,YU,ZR,ZM,ZW,**&ipsTechSelValues=100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145"),
    "TM3b": UrlSpec(url_template="https://api.ipstatsdc.deda.prd.web1.wipo.int/api/v1/public/ips-search/downloadCsv?selectedTab=trademark&indicator=36&reportType=15&fromYear={from_year}&toYear={to_year}&ipsOffSelValues=AF,OA,AP,AL,DZ,AD,AO,AG,AR,AM,AU,AT,AZ,BS,BH,BD,BB,BY,BE,BZ,BX,BJ,BT,BO,BQ,BA,BW,BR,BN,BG,BF,BI,CV,KH,CM,CA,CF,TD,CL,CN,HK,MO,CO,KM,CG,CK,CR,CI,HR,CU,CW,CY,CZ,CS,KP,CD,DK,DJ,DM,DO,EC,EG,SV,GQ,ER,EE,SZ,ET,EM,FJ,FI,FR,GA,GM,GE,DD,DE,GH,GR,GD,GT,GG,GN,GW,GY,HT,VA,HN,HU,IS,IN,ID,IR,IQ,IE,IL,IT,JM,JP,JO,KZ,KE,KI,KW,KG,LA,LV,LB,LS,LR,LY,LI,LT,LU,MG,MW,MY,MV,ML,MT,MH,MR,MU,MX,FM,MC,MN,ME,MA,MZ,MM,NA,NR,NP,NL,AN,NZ,NI,NE,NG,NU,MK,NO,OM,PK,PW,PA,PG,PY,PE,PH,PL,PT,QA,KR,MD,RO,RU,RW,KN,LC,VC,WS,SM,ST,SA,SN,RS,SC,SL,SG,SX,SK,SI,SB,SO,ZA,SS,SU,ES,LK,SD,SR,SE,CH,SY,TJ,TH,TL,TG,TO,TT,TN,TR,TM,TV,UG,UA,AE,GB,TZ,US,UY,UZ,VU,VE,VN,YE,YU,ZR,ZM,ZW&ipsOriSelValues=AF,AL,DZ,AD,AO,AG,AR,AM,AU,AT,AZ,BS,BH,BD,BB,BY,BE,BZ,BJ,BT,BO,BQ,BA,BW,BR,BN,BG,BF,BI,CV,KH,CM,CA,CF,TD,CL,CN,HK,MO,CO,KM,CG,CK,CR,CI,HR,CU,CW,CY,CZ,CS,KP,CD,DK,DJ,DM,DO,EC,EG,SV,GQ,ER,EE,SZ,ET,FJ,FI,FR,GA,GM,GE,DD,DE,GH,GR,GD,GT,GN,GW,GY,HT,VA,HN,HU,IS,IN,ID,IR,IQ,IE,IL,IT,JM,JP,JO,KZ,KE,KI,KW,KG,LA,LV,LB,LS,LR,LY,LI,LT,LU,MG,MW,MY,MV,ML,MT,MH,MR,MU,MX,FM,MC,MN,ME,MA,MZ,MM,NA,NR,NP,NL,AN,NZ,NI,NE,NG,NU,MK,NO,OM,PK,PW,PA,PG,PY,PE,PH,PL,PT,QA,KR,MD,RO,RU,RW,KN,LC,VC,WS,SM,ST,SA,SN,RS,SC,SL,SG,SX,SK,SI,SB,SO,ZA,SS,SU,ES,LK,SD,SR,SE,CH,SY,TJ,TH,TL,TG,TO,TT,TN,TR,TM,TV,UG,UA,AE,GB,TZ,US,UY,UZ,VU,VE,VN,YE,YU,ZR,ZM,ZW,**&ipsTechSelValues=100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145"),
    "TM4a": UrlSpec(url_template="https://api.ipstatsdc.deda.prd.web1.wipo.int/api/v1/public/ips-search/downloadCsv?selectedTab=trademark&indicator=37&reportType=15&fromYear={from_year}&toYear={to_year}&ipsOffSelValues=AF,OA,AP,AL,DZ,AD,AO,AG,AR,AM,AU,AT,AZ,BS,BH,BD,BB,BY,BE,BZ,BX,BJ,BT,BO,BQ,BA,BW,BR,BN,BG,BF,BI,CV,KH,CM,CA,CF,TD,CL,CN,HK,MO,CO,KM,CG,CK,CR,CI,HR,CU,CW,CY,CZ,CS,KP,CD,DK,DJ,DM,DO,EC,EG,SV,GQ,ER,EE,SZ,ET,EM,FJ,FI,FR,GA,GM,GE,DD,DE,GH,GR,GD,GT,GG,GN,GW,GY,HT,VA,HN,HU,IS,IN,ID,IR,IQ,IE,IL,IT,JM,JP,JO,KZ,KE,KI,KW,KG,LA,LV,LB,LS,LR,LY,LI,LT,LU,MG,MW,MY,MV,ML,MT,MH,MR,MU,MX,FM,MC,MN,ME,MA,MZ,MM,NA,NR,NP,NL,AN,NZ,NI,NE,NG,NU,MK,NO,OM,PK,PW,PA,PG,PY,PE,PH,PL,PT,QA,KR,MD,RO,RU,RW,KN,LC,VC,WS,SM,ST,SA,SN,RS,SC,SL,SG,SX,SK,SI,SB,SO,ZA,SS,SU,ES,LK,SD,SR,SE,CH,SY,TJ,TH,TL,TG,TO,TT,TN,TR,TM,TV,UG,UA,AE,GB,TZ,US,UY,UZ,VU,VE,VN,YE,YU,ZR,ZM,ZW&ipsOriSelValues=AF,AL,DZ,AD,AO,AG,AR,AM,AU,AT,AZ,BS,BH,BD,BB,BY,BE,BZ,BJ,BT,BO,BQ,BA,BW,BR,BN,BG,BF,BI,CV,KH,CM,CA,CF,TD,CL,CN,HK,MO,CO,KM,CG,CK,CR,CI,HR,CU,CW,CY,CZ,CS,KP,CD,DK,DJ,DM,DO,EC,EG,SV,GQ,ER,EE,SZ,ET,FJ,FI,FR,GA,GM,GE,DD,DE,GH,GR,GD,GT,GN,GW,GY,HT,VA,HN,HU,IS,IN,ID,IR,IQ,IE,IL,IT,JM,JP,JO,KZ,KE,KI,KW,KG,LA,LV,LB,LS,LR,LY,LI,LT,LU,MG,MW,MY,MV,ML,MT,MH,MR,MU,MX,FM,MC,MN,ME,MA,MZ,MM,NA,NR,NP,NL,AN,NZ,NI,NE,NG,NU,MK,NO,OM,PK,PW,PA,PG,PY,PE,PH,PL,PT,QA,KR,MD,RO,RU,RW,KN,LC,VC,WS,SM,ST,SA,SN,RS,SC,SL,SG,SX,SK,SI,SB,SO,ZA,SS,SU,ES,LK,SD,SR,SE,CH,SY,TJ,TH,TL,TG,TO,TT,TN,TR,TM,TV,UG,UA,AE,GB,TZ,US,UY,UZ,VU,VE,VN,YE,YU,ZR,ZM,ZW,**&ipsTechSelValues=100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145"),
    "TM4b": UrlSpec(url_template="https://api.ipstatsdc.deda.prd.web1.wipo.int/api/v1/public/ips-search/downloadCsv?selectedTab=trademark&indicator=38&reportType=15&fromYear={from_year}&toYear={to_year}&ipsOffSelValues=AF,OA,AP,AL,DZ,AD,AO,AG,AR,AM,AU,AT,AZ,BS,BH,BD,BB,BY,BE,BZ,BX,BJ,BT,BO,BQ,BA,BW,BR,BN,BG,BF,BI,CV,KH,CM,CA,CF,TD,CL,CN,HK,MO,CO,KM,CG,CK,CR,CI,HR,CU,CW,CY,CZ,CS,KP,CD,DK,DJ,DM,DO,EC,EG,SV,GQ,ER,EE,SZ,ET,EM,FJ,FI,FR,GA,GM,GE,DD,DE,GH,GR,GD,GT,GG,GN,GW,GY,HT,VA,HN,HU,IS,IN,ID,IR,IQ,IE,IL,IT,JM,JP,JO,KZ,KE,KI,KW,KG,LA,LV,LB,LS,LR,LY,LI,LT,LU,MG,MW,MY,MV,ML,MT,MH,MR,MU,MX,FM,MC,MN,ME,MA,MZ,MM,NA,NR,NP,NL,AN,NZ,NI,NE,NG,NU,MK,NO,OM,PK,PW,PA,PG,PY,PE,PH,PL,PT,QA,KR,MD,RO,RU,RW,KN,LC,VC,WS,SM,ST,SA,SN,RS,SC,SL,SG,SX,SK,SI,SB,SO,ZA,SS,SU,ES,LK,SD,SR,SE,CH,SY,TJ,TH,TL,TG,TO,TT,TN,TR,TM,TV,UG,UA,AE,GB,TZ,US,UY,UZ,VU,VE,VN,YE,YU,ZR,ZM,ZW&ipsOriSelValues=AF,AL,DZ,AD,AO,AG,AR,AM,AU,AT,AZ,BS,BH,BD,BB,BY,BE,BZ,BJ,BT,BO,BQ,BA,BW,BR,BN,BG,BF,BI,CV,KH,CM,CA,CF,TD,CL,CN,HK,MO,CO,KM,CG,CK,CR,CI,HR,CU,CW,CY,CZ,CS,KP,CD,DK,DJ,DM,DO,EC,EG,SV,GQ,ER,EE,SZ,ET,FJ,FI,FR,GA,GM,GE,DD,DE,GH,GR,GD,GT,GN,GW,GY,HT,VA,HN,HU,IS,IN,ID,IR,IQ,IE,IL,IT,JM,JP,JO,KZ,KE,KI,KW,KG,LA,LV,LB,LS,LR,LY,LI,LT,LU,MG,MW,MY,MV,ML,MT,MH,MR,MU,MX,FM,MC,MN,ME,MA,MZ,MM,NA,NR,NP,NL,AN,NZ,NI,NE,NG,NU,MK,NO,OM,PK,PW,PA,PG,PY,PE,PH,PL,PT,QA,KR,MD,RO,RU,RW,KN,LC,VC,WS,SM,ST,SA,SN,RS,SC,SL,SG,SX,SK,SI,SB,SO,ZA,SS,SU,ES,LK,SD,SR,SE,CH,SY,TJ,TH,TL,TG,TO,TT,TN,TR,TM,TV,UG,UA,AE,GB,TZ,US,UY,UZ,VU,VE,VN,YE,YU,ZR,ZM,ZW,**&ipsTechSelValues=100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145"),

    #fct_ip_flows_by_residency
    "TM1a_origin": UrlSpec(url_template="https://api.ipstatsdc.deda.prd.web1.wipo.int/api/v1/public/ips-search/downloadCsv?selectedTab=trademark&indicator=31&reportType=13&fromYear={from_year}&toYear={to_year}&ipsOriSelValues=AF,AL,DZ,AD,AO,AG,AR,AM,AU,AT,AZ,BS,BH,BD,BB,BY,BE,BZ,BJ,BT,BO,BQ,BA,BW,BR,BN,BG,BF,BI,CV,KH,CM,CA,CF,TD,CL,CN,HK,MO,CO,KM,CG,CK,CR,CI,HR,CU,CW,CY,CZ,CS,KP,CD,DK,DJ,DM,DO,EC,EG,SV,GQ,ER,EE,SZ,ET,FJ,FI,FR,GA,GM,GE,DD,DE,GH,GR,GD,GT,GN,GW,GY,HT,VA,HN,HU,IS,IN,ID,IR,IQ,IE,IL,IT,JM,JP,JO,KZ,KE,KI,KW,KG,LA,LV,LB,LS,LR,LY,LI,LT,LU,MG,MW,MY,MV,ML,MT,MH,MR,MU,MX,FM,MC,MN,ME,MA,MZ,MM,NA,NR,NP,NL,AN,NZ,NI,NE,NG,NU,MK,NO,OM,PK,PW,PA,PG,PY,PE,PH,PL,PT,QA,KR,MD,RO,RU,RW,KN,LC,VC,WS,SM,ST,SA,SN,RS,SC,SL,SG,SX,SK,SI,SB,SO,ZA,SS,SU,ES,LK,SD,SR,SE,CH,SY,TJ,TH,TL,TG,TO,TT,TN,TR,TM,TV,UG,UA,AE,GB,TZ,US,UY,UZ,VU,VE,VN,YE,YU,ZR,ZM,ZW&ipsTechSelValues=910,911,912,913,914"),
    "TM1b_origin": UrlSpec(url_template="https://api.ipstatsdc.deda.prd.web1.wipo.int/api/v1/public/ips-search/downloadCsv?selectedTab=trademark&indicator=32&reportType=13&fromYear={from_year}&toYear={to_year}&ipsOriSelValues=AF,AL,DZ,AD,AO,AG,AR,AM,AU,AT,AZ,BS,BH,BD,BB,BY,BE,BZ,BJ,BT,BO,BQ,BA,BW,BR,BN,BG,BF,BI,CV,KH,CM,CA,CF,TD,CL,CN,HK,MO,CO,KM,CG,CK,CR,CI,HR,CU,CW,CY,CZ,CS,KP,CD,DK,DJ,DM,DO,EC,EG,SV,GQ,ER,EE,SZ,ET,FJ,FI,FR,GA,GM,GE,DD,DE,GH,GR,GD,GT,GN,GW,GY,HT,VA,HN,HU,IS,IN,ID,IR,IQ,IE,IL,IT,JM,JP,JO,KZ,KE,KI,KW,KG,LA,LV,LB,LS,LR,LY,LI,LT,LU,MG,MW,MY,MV,ML,MT,MH,MR,MU,MX,FM,MC,MN,ME,MA,MZ,MM,NA,NR,NP,NL,AN,NZ,NI,NE,NG,NU,MK,NO,OM,PK,PW,PA,PG,PY,PE,PH,PL,PT,QA,KR,MD,RO,RU,RW,KN,LC,VC,WS,SM,ST,SA,SN,RS,SC,SL,SG,SX,SK,SI,SB,SO,ZA,SS,SU,ES,LK,SD,SR,SE,CH,SY,TJ,TH,TL,TG,TO,TT,TN,TR,TM,TV,UG,UA,AE,GB,TZ,US,UY,UZ,VU,VE,VN,YE,YU,ZR,ZM,ZW&ipsTechSelValues=910,911,912,913,914"),
    "TM1a_office": UrlSpec(url_template="https://api.ipstatsdc.deda.prd.web1.wipo.int/api/v1/public/ips-search/downloadCsv?selectedTab=trademark&indicator=31&reportType=11&fromYear={from_year}&toYear={to_year}&ipsOffSelValues=AF,OA,AP,AL,DZ,AD,AO,AG,AR,AM,AU,AT,AZ,BS,BH,BD,BB,BY,BE,BZ,BX,BJ,BT,BO,BQ,BA,BW,BR,BN,BG,BF,BI,CV,KH,CM,CA,CF,TD,CL,CN,HK,MO,CO,KM,CG,CK,CR,CI,HR,CU,CW,CY,CZ,CS,KP,CD,DK,DJ,DM,DO,EC,EG,SV,GQ,ER,EE,SZ,ET,EM,FJ,FI,FR,GA,GM,GE,DD,DE,GH,GR,GD,GT,GG,GN,GW,GY,HT,VA,HN,HU,IS,IN,ID,IR,IQ,IE,IL,IT,JM,JP,JO,KZ,KE,KI,KW,KG,LA,LV,LB,LS,LR,LY,LI,LT,LU,MG,MW,MY,MV,ML,MT,MH,MR,MU,MX,FM,MC,MN,ME,MA,MZ,MM,NA,NR,NP,NL,AN,NZ,NI,NE,NG,NU,MK,NO,OM,PK,PW,PA,PG,PY,PE,PH,PL,PT,QA,KR,MD,RO,RU,RW,KN,LC,VC,WS,SM,ST,SA,SN,RS,SC,SL,SG,SX,SK,SI,SB,SO,ZA,SS,SU,ES,LK,SD,SR,SE,CH,SY,TJ,TH,TL,TG,TO,TT,TN,TR,TM,TV,UG,UA,AE,GB,TZ,US,UY,UZ,VU,VE,VN,YE,YU,ZR,ZM,ZW,WD,R1,R2,R3,R4,R5,R6&ipsTechSelValues=900,901,902"),
    "TM1b_office": UrlSpec(url_template="https://api.ipstatsdc.deda.prd.web1.wipo.int/api/v1/public/ips-search/downloadCsv?selectedTab=trademark&indicator=32&reportType=11&fromYear={from_year}&toYear={to_year}&ipsOffSelValues=AF,OA,AP,AL,DZ,AD,AO,AG,AR,AM,AU,AT,AZ,BS,BH,BD,BB,BY,BE,BZ,BX,BJ,BT,BO,BQ,BA,BW,BR,BN,BG,BF,BI,CV,KH,CM,CA,CF,TD,CL,CN,HK,MO,CO,KM,CG,CK,CR,CI,HR,CU,CW,CY,CZ,CS,KP,CD,DK,DJ,DM,DO,EC,EG,SV,GQ,ER,EE,SZ,ET,EM,FJ,FI,FR,GA,GM,GE,DD,DE,GH,GR,GD,GT,GG,GN,GW,GY,HT,VA,HN,HU,IS,IN,ID,IR,IQ,IE,IL,IT,JM,JP,JO,KZ,KE,KI,KW,KG,LA,LV,LB,LS,LR,LY,LI,LT,LU,MG,MW,MY,MV,ML,MT,MH,MR,MU,MX,FM,MC,MN,ME,MA,MZ,MM,NA,NR,NP,NL,AN,NZ,NI,NE,NG,NU,MK,NO,OM,PK,PW,PA,PG,PY,PE,PH,PL,PT,QA,KR,MD,RO,RU,RW,KN,LC,VC,WS,SM,ST,SA,SN,RS,SC,SL,SG,SX,SK,SI,SB,SO,ZA,SS,SU,ES,LK,SD,SR,SE,CH,SY,TJ,TH,TL,TG,TO,TT,TN,TR,TM,TV,UG,UA,AE,GB,TZ,US,UY,UZ,VU,VE,VN,YE,YU,ZR,ZM,ZW,WD,R1,R2,R3,R4,R5,R6&ipsTechSelValues=900,901,902"),

    "TM2a_origin": UrlSpec(url_template="https://api.ipstatsdc.deda.prd.web1.wipo.int/api/v1/public/ips-search/downloadCsv?selectedTab=trademark&indicator=33&reportType=13&fromYear={from_year}&toYear={to_year}&ipsOriSelValues=AF,AL,DZ,AD,AO,AG,AR,AM,AU,AT,AZ,BS,BH,BD,BB,BY,BE,BZ,BJ,BT,BO,BQ,BA,BW,BR,BN,BG,BF,BI,CV,KH,CM,CA,CF,TD,CL,CN,HK,MO,CO,KM,CG,CK,CR,CI,HR,CU,CW,CY,CZ,CS,KP,CD,DK,DJ,DM,DO,EC,EG,SV,GQ,ER,EE,SZ,ET,FJ,FI,FR,GA,GM,GE,DD,DE,GH,GR,GD,GT,GN,GW,GY,HT,VA,HN,HU,IS,IN,ID,IR,IQ,IE,IL,IT,JM,JP,JO,KZ,KE,KI,KW,KG,LA,LV,LB,LS,LR,LY,LI,LT,LU,MG,MW,MY,MV,ML,MT,MH,MR,MU,MX,FM,MC,MN,ME,MA,MZ,MM,NA,NR,NP,NL,AN,NZ,NI,NE,NG,NU,MK,NO,OM,PK,PW,PA,PG,PY,PE,PH,PL,PT,QA,KR,MD,RO,RU,RW,KN,LC,VC,WS,SM,ST,SA,SN,RS,SC,SL,SG,SX,SK,SI,SB,SO,ZA,SS,SU,ES,LK,SD,SR,SE,CH,SY,TJ,TH,TL,TG,TO,TT,TN,TR,TM,TV,UG,UA,AE,GB,TZ,US,UY,UZ,VU,VE,VN,YE,YU,ZR,ZM,ZW&ipsTechSelValues=910,911,912,913,914"),
    "TM2b_origin": UrlSpec(url_template="https://api.ipstatsdc.deda.prd.web1.wipo.int/api/v1/public/ips-search/downloadCsv?selectedTab=trademark&indicator=34&reportType=13&fromYear={from_year}&toYear={to_year}&ipsOriSelValues=AF,AL,DZ,AD,AO,AG,AR,AM,AU,AT,AZ,BS,BH,BD,BB,BY,BE,BZ,BJ,BT,BO,BQ,BA,BW,BR,BN,BG,BF,BI,CV,KH,CM,CA,CF,TD,CL,CN,HK,MO,CO,KM,CG,CK,CR,CI,HR,CU,CW,CY,CZ,CS,KP,CD,DK,DJ,DM,DO,EC,EG,SV,GQ,ER,EE,SZ,ET,FJ,FI,FR,GA,GM,GE,DD,DE,GH,GR,GD,GT,GN,GW,GY,HT,VA,HN,HU,IS,IN,ID,IR,IQ,IE,IL,IT,JM,JP,JO,KZ,KE,KI,KW,KG,LA,LV,LB,LS,LR,LY,LI,LT,LU,MG,MW,MY,MV,ML,MT,MH,MR,MU,MX,FM,MC,MN,ME,MA,MZ,MM,NA,NR,NP,NL,AN,NZ,NI,NE,NG,NU,MK,NO,OM,PK,PW,PA,PG,PY,PE,PH,PL,PT,QA,KR,MD,RO,RU,RW,KN,LC,VC,WS,SM,ST,SA,SN,RS,SC,SL,SG,SX,SK,SI,SB,SO,ZA,SS,SU,ES,LK,SD,SR,SE,CH,SY,TJ,TH,TL,TG,TO,TT,TN,TR,TM,TV,UG,UA,AE,GB,TZ,US,UY,UZ,VU,VE,VN,YE,YU,ZR,ZM,ZW&ipsTechSelValues=910,911,912,913,914"),
    "TM2a_office": UrlSpec(url_template="https://api.ipstatsdc.deda.prd.web1.wipo.int/api/v1/public/ips-search/downloadCsv?selectedTab=trademark&indicator=33&reportType=11&fromYear={from_year}&toYear={to_year}&ipsOffSelValues=AF,OA,AP,AL,DZ,AD,AO,AG,AR,AM,AU,AT,AZ,BS,BH,BD,BB,BY,BE,BZ,BX,BJ,BT,BO,BQ,BA,BW,BR,BN,BG,BF,BI,CV,KH,CM,CA,CF,TD,CL,CN,HK,MO,CO,KM,CG,CK,CR,CI,HR,CU,CW,CY,CZ,CS,KP,CD,DK,DJ,DM,DO,EC,EG,SV,GQ,ER,EE,SZ,ET,EM,FJ,FI,FR,GA,GM,GE,DD,DE,GH,GR,GD,GT,GG,GN,GW,GY,HT,VA,HN,HU,IS,IN,ID,IR,IQ,IE,IL,IT,JM,JP,JO,KZ,KE,KI,KW,KG,LA,LV,LB,LS,LR,LY,LI,LT,LU,MG,MW,MY,MV,ML,MT,MH,MR,MU,MX,FM,MC,MN,ME,MA,MZ,MM,NA,NR,NP,NL,AN,NZ,NI,NE,NG,NU,MK,NO,OM,PK,PW,PA,PG,PY,PE,PH,PL,PT,QA,KR,MD,RO,RU,RW,KN,LC,VC,WS,SM,ST,SA,SN,RS,SC,SL,SG,SX,SK,SI,SB,SO,ZA,SS,SU,ES,LK,SD,SR,SE,CH,SY,TJ,TH,TL,TG,TO,TT,TN,TR,TM,TV,UG,UA,AE,GB,TZ,US,UY,UZ,VU,VE,VN,YE,YU,ZR,ZM,ZW,WD,R1,R2,R3,R4,R5,R6&ipsTechSelValues=900,901,902"),
    "TM2b_office": UrlSpec(url_template="https://api.ipstatsdc.deda.prd.web1.wipo.int/api/v1/public/ips-search/downloadCsv?selectedTab=trademark&indicator=34&reportType=11&fromYear={from_year}&toYear={to_year}&ipsOffSelValues=AF,OA,AP,AL,DZ,AD,AO,AG,AR,AM,AU,AT,AZ,BS,BH,BD,BB,BY,BE,BZ,BX,BJ,BT,BO,BQ,BA,BW,BR,BN,BG,BF,BI,CV,KH,CM,CA,CF,TD,CL,CN,HK,MO,CO,KM,CG,CK,CR,CI,HR,CU,CW,CY,CZ,CS,KP,CD,DK,DJ,DM,DO,EC,EG,SV,GQ,ER,EE,SZ,ET,EM,FJ,FI,FR,GA,GM,GE,DD,DE,GH,GR,GD,GT,GG,GN,GW,GY,HT,VA,HN,HU,IS,IN,ID,IR,IQ,IE,IL,IT,JM,JP,JO,KZ,KE,KI,KW,KG,LA,LV,LB,LS,LR,LY,LI,LT,LU,MG,MW,MY,MV,ML,MT,MH,MR,MU,MX,FM,MC,MN,ME,MA,MZ,MM,NA,NR,NP,NL,AN,NZ,NI,NE,NG,NU,MK,NO,OM,PK,PW,PA,PG,PY,PE,PH,PL,PT,QA,KR,MD,RO,RU,RW,KN,LC,VC,WS,SM,ST,SA,SN,RS,SC,SL,SG,SX,SK,SI,SB,SO,ZA,SS,SU,ES,LK,SD,SR,SE,CH,SY,TJ,TH,TL,TG,TO,TT,TN,TR,TM,TV,UG,UA,AE,GB,TZ,US,UY,UZ,VU,VE,VN,YE,YU,ZR,ZM,ZW,WD,R1,R2,R3,R4,R5,R6&ipsTechSelValues=900,901,902"),






}

In [16]:
# Main execution

if __name__ == "__main__":
    if not URL_TEMPLATES:
        print("WARNING: URL_TEMPLATES is empty. Please paste your URLs in Section 2.")
    
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    session = get_session()
    prime_cookies(session)

    print(f"Starting download for years {from_year} to {to_year}...")
    print(f"Saving to: {OUTPUT_DIR.absolute()}\n")

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {
            executor.submit(download_task, session, key, url, OUTPUT_DIR): key 
            for key, url in URL_TEMPLATES.items()
        }

        for future in as_completed(futures):
            key = futures[future]
            try:
                result = future.result()
                if result:
                    print(result)
            except Exception as e:
                print(f" Error downloading {key}: {e}")

    print("\nAll tasks finished.")

Priming cookies (visiting homepage)...


Starting download for years 1980 to 2024...
Saving to: C:\Users\shresthn\OneDrive - ISED-ISDE\Desktop\or_country_profiles-main\data\raw\wipo\ip_indicators

Skipped (Exists): industrial_1b- Applications via the Hague system_Count by filing office and applicant's origin_1980_2024.csv
Skipped (Exists): industrial_2a- Registrations for direct applications_Count by filing office and applicant's origin_1980_2024.csv
Skipped (Exists): industrial_1a- Direct applications_Count by filing office and applicant's origin_1980_2024.csv
Skipped (Exists): industrial_2b- Registrations for applications via the Hague system_Count by filing office and applicant's origin_1980_2024.csv
Skipped (Exists): industrial_7 - Design registrations in force (by office)_Total count by filing office_1980_2024.csv
Skipped (Exists): industrial_3a- Direct applications by class_Count by filing office and applicant's origin_1980_2024.csv
Skipped (Exists): industrial_3b- Applications via the Hague system by class_Count by fil